# Suite DIA — The Python, Matlab, Excel, Latex, Ccpp and SystemVerilog dialects

Each dialect models a language's expressions: its kinds, its vocabulary and the domains of its values, `render`
to its source text, an evaluator with the language's own rules, and a scope that resolves names, imports and
references as the language does. Python evaluates by Python's rules, with NumPy as one more module to import;
MATLAB is two-valued over doubles, logicals and strings, with functions from the path and packages; Excel coerces,
computes with error values, and reads cells from workbooks. Latex writes notation, and has no evaluator. Ccpp evaluates by C's
rules over its arithmetic types, and SystemVerilog by IEEE 1800's over 4-state vectors and reals. What every dialect shares is suite FRM; the Basic
dialect is suites EXP and EVL.

In [ ]:
import math

import numpy as np

import mbse.Expressions.Dialects.Latex as LatexPackage
from mbse.Expressions.Dialects.Basic import Domains as BD
from mbse.Expressions.Dialects.Ccpp import Domains as CD, Evaluators as CV, Expressions as C, Text as CText
from mbse.Expressions.Dialects.Excel import Domains as XD, Evaluators as XV, Expressions as X, Text as XText
from mbse.Expressions.Dialects.Latex import Domains as LD, Expressions as L, Text as LText
from mbse.Expressions.Dialects.Matlab import Domains as MD, Evaluators as MV, Expressions as M, Text as MText
from mbse.Expressions.Dialects.Python import Domains as PD, Evaluators as PV, Expressions as P, Text as PText
from mbse.Expressions.Dialects.SystemVerilog import Domains as SVD, Evaluators as SVV, Expressions as SV, Text as SVText
from mbse.Expressions.Framework import Domains as D, Errors as FE, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Proxies, Schemas as S
from support import raises, text
from mbse.Expressions import register

store = register(Proxies.OfStore())  # holds every dialect's meta-schemas

Person = S.OfObject.Builder().name("Person").ref().properties(text("name"), text("age", int), text("email")).create()
store.register(Person)
ann = store.Person().name("Ann").age(30).email("ann@example.com").create()
bob = store.Person().name("Bob").age(70).create()  # no email

## DIA-01 · Python: expressions as `ast` has them, with their imports; rendered and parsed as source

In [ ]:
t, a, b, c, d, x, y, z = (P.name(n) for n in "tabcdxyz")
for built, source in [  # written with constructors, as source
    (P.import_("numpy", P.importfrom("numpy.ma", "getmaskarray", P.call(
        "np.logical_and", P.call("np.greater_equal", P.subscript(t, "age"), 18),
        P.call("np.logical_not", P.call("getmaskarray", P.subscript(t, "email"))))), alias="np"),
     "import numpy as np\nfrom numpy.ma import getmaskarray\n"
     "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))"),
    (P.let_("a", t, P.boolop("and", P.compare(">", P.attribute(a, "b"), 1), P.unaryop("not", P.attribute(a, "c")))),
     "(lambda a: a.b > 1 and not a.c)(t)"),
    (P.ifexp(P.compare("<", a, b), x, P.unaryop("-", P.binop("**", y, 2))), "x if a < b else -y ** 2"),
    (P.ifexp(b, P.ifexp(a, x, y), z), "(x if a else y) if b else z"),
    (P.ifexp(a, x, P.ifexp(b, y, z)), "x if a else y if b else z"),
    (P.binop("-", P.binop("-", P.binop("**", 2, P.unaryop("-", 1)), P.binop("-", a, b)), c), "2 ** -1 - (a - b) - c"),
    (P.boolop("or", P.boolop("and", a, b), P.boolop("and", c, P.unaryop("not", d))), "a and b or c and not d"),
    (P.boolop("and", P.boolop("or", a, b), P.boolop("or", c, d)), "(a or b) and (c or d)"),
    (P.compare("!=", P.unaryop("not", P.compare("==", a, b)), P.compare("<=", c, d)), "(not a == b) != (c <= d)"),
    (P.call(P.call("f", P.name("g")), P.subscript(P.attribute(x, "y"), "z"), P.unaryop("+", 1), b"\x00", True, "it's"),
     "f(g)(x.y['z'], +1, b'\\x00', True, \"it's\")"),
    (P.import_("a", P.import_("b", P.importfrom("d", "e", P.importfrom("d", "f", 1, alias="g")), alias="c")),
     "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1"),
    (P.call("all", P.generator("p", P.attribute(t, "ports"), P.compare(">", P.attribute(P.name("p"), "pin"), 0))),
     "all(p.pin > 0 for p in t.ports)"),
    (P.call("sum", P.generator("p", x, 1, P.compare("in", P.name("p"), y), P.compare("not in", P.name("p"), z))),
     "sum(1 for p in x if p in y if p not in z)"),
    (P.binop("+", P.index(x, P.binop("-", a, 1)), P.call("len", P.call("set", x))), "x[a - 1] + len(set(x))"),
    (P.call("max", P.generator("v", P.ifexp(a, x, y), P.ifexp(b, P.name("v"), z), P.boolop("or", a, b)), 1),
     "max((v if b else z for v in (x if a else y) if a or b), 1)"),
]:
    assert PText.ToText(built) == source, (PText.ToText(built), source)
    assert F.same(PText.FromText(source), built), source  # Python reads it back (parse is Python's only)
for source in [
    "import numpy as np\nfrom numpy.ma import getmaskarray\nnp.logical_and(np.greater_equal(t['age'], 18), "
    "np.logical_not(getmaskarray(t['email'])))",
    "import numpy.ma\nnumpy.ma.masked",
    "(lambda a: a.b > 1 and not a.c)(t)",
    "x if a < b else -y ** 2",
    "(x if a else y) if b else z",
    "(-1) ** 2 + 3 * (4 - 5) // 2 % 7 / 1.5",
    "2 ** -1 - (a - b) - c",
    "a and b or c and not d",
    "(a or b) and (c or d)",
    "not (a == b) != (c <= d)",
    "f(g)(x.y['z'], +1, b'\\x00', True, 'it''s')",
]:
    parsed = PText.FromText(source)
    assert PText.ToText(parsed) == PText.ToText(PText.FromText(PText.ToText(parsed))), source  # render and parse are inverses
    assert F.same(PText.FromText(PText.ToText(parsed)), parsed), source
assert PText.ToText(PText.FromText("import numpy.ma\nnumpy.ma.masked")) == "import numpy.ma\nnumpy.ma.masked"
assert PText.ToText(PText.FromText("x if a else y if b else z")) == "x if a else y if b else z"
assert PText.ToText(PText.FromText("(x if a else y) if b else z")) == "(x if a else y) if b else z"
assert PText.ToText(PText.FromText("import a, b as c\nfrom d import e, f as g\n1")) == (
    "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1")
assert [PText.ToText(P.constant(v)) for v in [math.nan, math.inf, -math.inf, -2]] == [
    "float('nan')", "float('inf')", "-float('inf')", "-2"]
assert PText.ToText(P.binop("**", -2, 2)) == "(-2) ** 2" and PText.ToText(P.unaryop("-", P.binop("**", 2, 2))) == "-2 ** 2"
assert PText.ToText(P.call("np.ma.getmaskarray", P.name("x"))) == "np.ma.getmaskarray(x)"  # a dotted function
built = P.importfrom("m", "f", P.let_("a", P.subscript(P.name("t"), "k"), P.ifexp(
    P.boolop("and", P.compare("<", P.name("a"), 1), P.unaryop("not", P.attribute(P.name("a"), "x"))),
    P.binop("+", P.call("f", P.name("a")), 2.5), P.constant(b""))), alias="g")
assert F.same(built, PText.FromText("from m import f as g\n(lambda a: f(a) + 2.5 if a < 1 and not a.x else b'')(t['k'])"))
assert PText.ToText(P.attribute(P.binop("+", 1, 2), "real")) == "(1 + 2).real"
counted = P.generator("p", P.name("ps"), P.name("p"), P.name("q"))  # p is bound in the element and the conditions only
assert SY.free(counted) == {"ps", "q"} and counted.validate(bound={"ps", "q"}) == []
assert P.generator("1p", P.name("ps"), 1).validate(bound={"ps"}) == ["a generator's name must be an identifier, got '1p'"]
assert P.generator("p", P.name("p"), 1).validate() == ["iterable: name 'p' is not bound"]
chain = PText.FromText("a < b <= c")  # a < b and b <= c, with b shared
assert PText.ToText(chain) == "a < b and b <= c"
assert chain.form().arguments[0].form().arguments[1] is chain.form().arguments[1].form().arguments[0]
with raises(ValueError, match="an import can only enclose the whole expression"):
    PText.ToText(P.binop("+", 1, P.import_("math", 2)))
for source, message in [
    ("f(x=1)", "cannot parse 'f(x=1)': keyword arguments are not supported"),
    ("(lambda a, b: a)(1, 2)", "cannot parse '(lambda a, b: a)(1, 2)': a let binds one name"),
    ("(lambda a=1: a)(2)", "a let binds one name"),
    ("x[0:1]", "cannot parse 'x[0:1]': slices are not supported"),
    ("(a for b in c for d in e)", "cannot parse '(a for b in c for d in e)': a generator has one for, over a name"),
    ("(a for (b, c) in d)", "a generator has one for, over a name"),
    ("None", "cannot parse 'None': only native constants are supported"),
    ("a @ b", "cannot parse 'a @ b': not supported in an expression"),
    ("a is b", "cannot parse 'a is b': not supported in an expression"),
    ("{1}", "cannot parse '{1}': not supported in an expression"),
    ("x = 1\nx", "cannot parse 'x = 1': only imports may precede the expression"),
    ("from . import a\na", "cannot parse 'from . import a': only imports may precede the expression"),
    ("from m import *\na", "cannot parse 'from m import *': import * binds names that cannot be known"),
]:
    with raises(ValueError, match=message):
        PText.FromText(source)
for source in ["", "import math"]:
    with raises(ValueError, match="the source must end with an expression"):
        PText.FromText(source)
assert P.import_("numpy.ma", 1).binds() == ("numpy",) and P.importfrom("m", "n", 1, alias="k").binds() == ("k",)
assert P.import_("not a module", 1).validate() == ["an import's module must be a dotted name, got 'not a module'"]
assert P.import_("m", 1, alias="a b").validate() == ["an import's alias must be an identifier, got 'a b'"]
assert P.importfrom("m.", "x y", 1).validate() == ["an importfrom's module must be a dotted name, got 'm.'",
                                                    "an importfrom's name must be an identifier, got 'x y'"]
assert P.importfrom("m", "n", 1, alias="").validate() == ["an importfrom needs an alias"]  # empty, not absent
assert P.let_("a b", 1, 2).validate() == ["a let's name must be an identifier, got 'a b'"]
assert P.attribute(P.name("x"), "1").validate(bound={"x"}) == ["an attribute must be an identifier, got '1'"]
assert P.importfrom("m", "n", P.name("n")).validate() == [] and P.DIALECT.kinds()["import"](7, None, P.constant(1)).validate() == [
    "an import's module must be a str, got int"]
assert P.binop("@", 1, 2).validate(core=True) == ["'@' is not a core operation"]
assert P.DIALECT.infer(PText.FromText("1 + 2.5")) is PD.Float and P.DIALECT.infer(PText.FromText("True + 1")) is PD.Int
assert P.DIALECT.infer(PText.FromText("'a' + 'b'")) is PD.Str and P.DIALECT.infer(PText.FromText("-2.0")) is PD.Float
assert P.DIALECT.infer(PText.FromText("2 ** 3")).name() == "int | float" and P.DIALECT.infer(PText.FromText("not 1")) is PD.Bool
assert P.DIALECT.infer(PText.FromText("1 < 2.5")) is PD.Bool and P.DIALECT.infer(PText.FromText("x.a"), {"x": D.Anything}) is D.Anything
with raises(TypeError, match="< cannot take (str, int)"):
    P.DIALECT.infer(PText.FromText("'a' < 1"))
assert PD.of(b"") is PD.Bytes and PD.Numeric.includes(PD.Bool)

## DIA-02 · Python evaluates by Python's rules, in a scope that allowlists what an expression may reach

In [ ]:
def ev(source, variables=None, **scope):
    return PV.OfAny(PText.FromText(source), PV.Scope(variables, **scope))


assert ev("0 or 'x'") == "x" and ev("'' and 1") == "" and ev("1 and 2") == 2  # an operand, not a bool
assert ev("1 == 1.0") is True and ev("True + 1") == 2 and ev("7 // 2 % 3 - 2 ** 2 / 4") == -1.0
assert ev("a if a > 1 else missing", {"a": 2}) == 2 and ev("(lambda a: a * a)(3)") == 9
assert ev("t.age", {"t": ann}) == 30 and ev("hasattr(t, 'email')", {"t": bob}) is False  # properties are attributes
assert ev("getattr(t, 'name')", {"t": ann}) == "Ann" and ev("t['x']", {"t": {"x": 1}}) == 1
assert ev("hasattr(t, '_secret')", {"t": ann}) is False
assert ev("abs(-2) + len('abc') + max(1, 2)") == 7 and ev("f(2)", {"f": lambda x: x + 1}) == 3
with raises(AttributeError, match="'Person' object has no attribute 'email'"):
    ev("t.email", {"t": bob})
with raises(AttributeError, match="attribute '__class__' is private"):
    ev("t.__class__", {"t": ann})
with raises(NameError, match="name 'x' is not defined"):
    ev("x")
with raises(NameError, match="name 'len' is not defined"):
    ev("len('a')", builtins={})


def shout(text):
    return text.upper()


with raises(TypeError, match="calling 'shout' is not allowed by the scope"):
    ev("t['f']('a')", {"t": {"f": shout}})  # only what the scope provides, not what values hold
with raises(TypeError, match="calling 'int' is not allowed by the scope"):
    ev("t()", {"t": 3})

numpy = {"numpy": np}
columns = {"age": np.array([30, 10, 70]), "email": np.ma.array(["a", "b", "c"], mask=[False, False, True])}
rule = ("import numpy as np\nfrom numpy.ma import getmaskarray\n"
        "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))")
assert ev(rule, {"t": columns}, modules=numpy).tolist() == [True, False, False]  # vectorized, with masks
assert ev("import numpy.ma\nnumpy.ma.masked", modules=numpy) is np.ma.masked  # binds numpy
assert ev("import numpy.ma as ma\nma.masked", modules=numpy) is np.ma.masked and ev("import numpy\nnumpy.pi",
                                                                                    modules=numpy) == math.pi
with raises(ImportError, match="import of 'numpy' is not allowed by the scope"):
    ev(rule, {"t": columns})  # nothing is imported unless the scope allows it
with raises(ImportError, match="import of 'os' is not allowed by the scope"):
    ev("import os\nos.getcwd()", modules=numpy)
with raises(ImportError, match="No module named 'numpy.nope'"):
    ev("import numpy.nope\n1", modules=numpy)
with raises(ImportError, match="No module named 'numpy._core'"):
    ev("import numpy._core\n1", modules=numpy)
with raises(ImportError, match="cannot import name 'nope' from 'numpy'"):
    ev("from numpy import nope\n1", modules=numpy)
with raises(ImportError, match="cannot import name '_core' from 'numpy'"):
    ev("from numpy import _core\n1", modules=numpy)
assert PV.OfAny(PText.FromText("x + 1"), {"x": 1}) == 2  # a mapping is the variables of a scope

# Python's rules, case by case. Here they run against Python itself; the TypeScript suite runs the same table against
# its model of them.
import math as math_module

n, NAN, INF = P.name, float("nan"), float("inf")
modules = {"math": math_module, "numpy": np}
Squad = S.OfObject.Builder().name("Squad").ref().properties(lambda p: p.name("ranks").of(
    lambda t: t.as_indexed(lambda i: i.of(S.OfNative.Data(int))))).create()
store.register(Squad)
squad = store.Squad().ranks([1, 2, 3]).create()  # a list property is a list of its values
VALUES = [  # (expression, variables, value): the value and its type
    (P.unaryop("not", n("v")), {"v": None}, True), (P.unaryop("not", 0.0), {}, True), (P.unaryop("not", ""), {}, True),
    (P.unaryop("not", b""), {}, True), (P.unaryop("not", n("v")), {"v": []}, True), (P.unaryop("not", n("v")), {"v": {}}, True),
    (P.unaryop("not", NAN), {}, False), (P.unaryop("not", n("v")), {"v": {"k": 1}}, False),
    (P.compare("<", 1, 1.5), {}, True), (P.compare(">", 2, 1.5), {}, True), (P.compare("==", 1, 1.5), {}, False),
    (P.compare("==", P.binop("+", P.binop("**", 2, 60), 1), P.call("float", P.binop("**", 2, 60))), {}, False),
    (P.compare("<", 1, INF), {}, True), (P.compare(">", 1, -INF), {}, True), (P.compare("<", 1, NAN), {}, False),
    (P.compare("<", 1.5, 2), {}, True), (P.compare("==", NAN, NAN), {}, False), (P.compare("==", 3, 3.0), {}, True),
    (P.compare("<", 3, 3.5), {}, True), (P.compare("<", 2.5, 1.5), {}, False), (P.compare("<", True, 2), {}, True),
    (P.compare("<", "a", "b"), {}, True), (P.compare("<", b"a", b"b"), {}, True), (P.compare("==", b"a", b"a"), {}, True),
    (P.compare("!=", "a", 1), {}, True),
    (P.binop("+", "ab", "c"), {}, "abc"), (P.binop("+", b"a", b"b"), {}, b"ab"), (P.binop("*", "ab", 2), {}, "abab"),
    (P.binop("*", 2, b"x"), {}, b"xx"), (P.binop("*", "ab", -1), {}, ""), (P.binop("//", -7, 2), {}, -4),
    (P.binop("%", -7, 2), {}, 1), (P.binop("%", 7, -2), {}, -1), (P.binop("%", -7.5, 2), {}, 0.5), (P.binop("%", 7.5, 2), {}, 1.5),
    (P.binop("//", 7.5, 2), {}, 3.0), (P.binop("**", 2, -1), {}, 0.5), (P.binop("**", 2.0, 3), {}, 8.0),
    (P.binop("+", 1.5, 1), {}, 2.5), (P.binop("*", 1.5, 2), {}, 3.0), (P.binop("-", 3, 1.5), {}, 1.5), (P.binop("/", 1, 2), {}, 0.5),
    (P.unaryop("-", True), {}, -1), (P.unaryop("+", True), {}, 1), (P.unaryop("-", 1.5), {}, -1.5), (P.unaryop("+", 1.5), {}, 1.5),
    (P.call("abs", True), {}, 1), (P.call("abs", -1.5), {}, 1.5), (P.call("bool", ""), {}, False), (P.call("float", 1), {}, 1.0),
    (P.call("float", " 1e3 "), {}, 1000.0), (P.call("float", "inf"), {}, INF), (P.call("float", "-inf"), {}, -INF),
    (P.call("str", P.call("float", "nan")), {}, "nan"), (P.call("int", " 42 "), {}, 42), (P.call("int", 2.7), {}, 2),
    (P.call("int", -2.7), {}, -2), (P.call("int", True), {}, 1), (P.call("len", b"ab"), {}, 2),
    (P.call("len", n("v")), {"v": [1, 2]}, 2), (P.call("len", n("v")), {"v": {"a": 1}}, 1), (P.call("len", "héllo"), {}, 5),
    (P.call("max", 1, 2.5), {}, 2.5), (P.call("min", 3, 1, 2), {}, 1), (P.call("round", 2.5), {}, 2),
    (P.call("round", 3.5), {}, 4), (P.call("round", -0.5), {}, 0), (P.call("round", 7), {}, 7), (P.call("str", 1.0), {}, "1.0"),
    (P.call("str", b"x"), {}, "b'x'"), (P.call("str", n("v")), {"v": None}, "None"), (P.call("str", True), {}, "True"),
    (P.call("str", 7), {}, "7"), (P.call("str", "s"), {}, "s"), (P.import_("math", P.call("math.floor", 2.5)), {}, 2),
    (P.unaryop("-", False), {}, 0), (P.unaryop("not", n("v")), {"v": ann}, False), (P.compare("==", 1.5, 1.5), {}, True),
    (P.binop("/", 1.5, 2), {}, 0.75), (P.binop("&", 12, 10), {}, 8), (P.binop("|", 12, True), {}, 13),
    (P.binop("^", True, True), {}, False), (P.binop("&", True, False), {}, False), (P.binop("|", False, True), {}, True),
    (P.binop("<<", True, 3), {}, 8), (P.binop(">>", -7, 1), {}, -4), (P.unaryop("~", 5), {}, -6),
    (P.binop("|", P.binop("**", 2, 70), 1), {}, 2 ** 70 + 1),
    (P.call("all", P.generator("p", n("v"), P.compare(">", n("p"), 0))), {"v": [1, 2]}, True),
    (P.call("all", P.generator("p", n("v"), n("p"))), {"v": []}, True), (P.call("any", n("v")), {"v": [0, 3]}, True),
    (P.call("any", P.generator("p", n("v"), n("p"))), {"v": [0, ""]}, False),
    (P.call("sum", P.generator("p", n("v"), 1, P.compare(">", n("p"), 0))), {"v": [1, -1, 2]}, 2),
    (P.call("sum", P.generator("c", "abc", 1, P.compare("!=", n("c"), "b"), P.compare("!=", n("c"), "c"))), {}, 1),
    (P.call("sum", n("v")), {"v": [0.1] * 10}, 1.0), (P.call("sum", n("v")), {"v": [1, 0.1, 0.2, 0.3]}, 1.6),
    (P.call("sum", n("v")), {"v": [1e16, 1, 1]}, 1.0000000000000002e16), (P.call("sum", n("v"), True), {"v": [0.1] * 10}, 2.000000000000001),
    (P.call("sum", n("v")), {"v": [True, True]}, 2), (P.call("sum", n("v")), {"v": []}, 0), (P.call("sum", n("v"), 1.5), {"v": []}, 1.5),
    (P.call("sum", n("v")), {"v": [2 ** 62, 2 ** 62, 0.5]}, 9.223372036854776e18), (P.call("sum", n("v")), {"v": [2 ** 63 - 1, 1]}, 2 ** 63),
    (P.call("sum", n("v")), {"v": [-654.25, -2 ** 63 - 1, -9, 2 ** 63 + 1]}, -663.25),
    (P.call("sum", n("v"), 2 ** 64), {"v": [1, 0.5]}, 1.8446744073709552e19),
    (P.call("min", n("v")), {"v": [3, 1, 2]}, 1), (P.call("max", P.generator("p", n("v"), P.binop("*", n("p"), 2))), {"v": [1, 2.5]}, 5.0),
    (P.call("max", "abc"), {}, "c"), (P.call("min", P.call("set", n("v"))), {"v": [2, 1]}, 1),
    (P.call("len", P.call("set", n("v"))), {"v": [1, 1.0, True, 2]}, 2), (P.call("len", P.call("set")), {}, 0),
    (P.unaryop("not", P.call("set")), {}, True),
    (P.compare("in", 2, n("v")), {"v": [1, 2]}, True), (P.compare("in", "b", "abc"), {}, True),
    (P.compare("in", "k", n("v")), {"v": {"k": 1}}, True), (P.compare("not in", 3, n("v")), {"v": [1, 2]}, True),
    (P.compare("in", 1.0, n("v")), {"v": [1]}, True), (P.compare("in", 98, b"ab"), {}, True), (P.compare("in", b"b", b"ab"), {}, True),
    (P.compare("in", b"ba", b"ab"), {}, False), (P.compare("in", n("w"), n("v")), {"v": [[1]], "w": [1]}, True),
    (P.compare("in", 1, P.call("set", n("v"))), {"v": [1]}, True), (P.compare("in", 3, P.generator("p", n("v"), n("p"))), {"v": [1, 2]}, False),
    (P.index(n("v"), 1), {"v": [10, 20]}, 20), (P.index(n("v"), -1), {"v": [10, 20]}, 20), (P.index("abc", 1), {}, "b"),
    (P.index(b"ab", 0), {}, 97), (P.index(n("v"), True), {"v": [10, 20]}, 20), (P.index(n("v"), 1.0), {"v": {1: "x"}}, "x"),
    (P.index(n("v"), "k"), {"v": {"k": 1}}, 1), (P.call("all", P.index(n("v"), 0)), {"v": [[1, 1]]}, True),
    (P.call("sum", b"\x01\x02"), {}, 3), (P.call("max", b"ab"), {}, 98), (P.call("len", P.call("set", n("v"))), {"v": {"a": 1, "b": 2}}, 2),
    (P.index(n("v"), "b"), {"v": {"a": 1, "b": 2}}, 2),
    (P.call("sum", P.attribute(n("v"), "ranks")), {"v": squad}, 6), (P.index(P.attribute(n("v"), "ranks"), -1), {"v": squad}, 3),
]
for expression, variables, expected in VALUES:
    value = PV.OfAny(expression, PV.Scope(variables, modules=modules))
    assert value == expected and type(value) is type(expected), (PText.ToText(expression), value, expected)
ERRORS = [  # (expression, variables, error, message)
    (P.compare("<", "a", 1), {}, TypeError, "'<' not supported between instances of 'str' and 'int'"),
    (P.binop("*", "a", 1.5), {}, TypeError, "can't multiply sequence by non-int of type 'float'"),
    (P.binop("-", "a", 1), {}, TypeError, "unsupported operand type(s) for -: 'str' and 'int'"),
    (P.binop("&", 1.5, 1), {}, TypeError, "unsupported operand type(s) for &: 'float' and 'int'"),
    (P.binop("<<", 1, -1), {}, ValueError, "negative shift count"), (P.binop(">>", 1, -1), {}, ValueError, "negative shift count"),
    (P.unaryop("~", 1.5), {}, TypeError, "bad operand type for unary ~: 'float'"),
    (P.binop("/", 1, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("//", 1.5, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("%", 1, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("**", 0.0, -1), {}, ZeroDivisionError, "zero to a negative power"),
    (P.unaryop("-", "a"), {}, TypeError, "bad operand type for unary -: 'str'"),
    (P.call("abs", "a"), {}, TypeError, "bad operand type for abs(): 'str'"),
    (P.call("float", "x"), {}, ValueError, "could not convert string to float: 'x'"),
    (P.call("int", "x"), {}, ValueError, "invalid literal for int() with base 10: 'x'"),
    (P.call("int", NAN), {}, ValueError, "cannot convert float NaN to integer"),
    (P.call("int", INF), {}, OverflowError, "cannot convert float infinity to integer"),
    (P.call("len", 1), {}, TypeError, "object of type 'int' has no len()"),
    (P.call("max", 1, "a"), {}, TypeError, "'>' not supported between instances of 'str' and 'int'"),
    (P.call("round", "x"), {}, TypeError, "type str doesn't define __round__ method"),
    (P.call("float", n("v")), {"v": None}, TypeError, "float() argument must be a string or a real number, not 'NoneType'"),
    (P.call("int", n("v")), {"v": None}, TypeError,
     "int() argument must be a string, a bytes-like object or a real number, not 'NoneType'"),
    (P.import_("numpy", P.attribute(n("numpy"), "nope")), {}, AttributeError, "module 'numpy' has no attribute 'nope'"),
    (P.subscript(n("t"), "y"), {"t": {"x": 1}}, KeyError, "'y'"),
    (P.subscript(n("t"), "x"), {"t": 3}, TypeError, "'int' object is not subscriptable"),
    (P.call(P.subscript(n("t"), "f")), {"t": {"f": lambda: 1}}, TypeError, "calling '<lambda>' is not allowed by the scope"),
    (P.call("len", P.generator("p", n("v"), n("p"))), {"v": [1]}, TypeError, "object of type 'generator' has no len()"),
    (P.call("sum", n("v")), {"v": ["a"]}, TypeError, "unsupported operand type(s) for +: 'int' and 'str'"),
    (P.call("sum", n("v")), {"v": [0.5, "a"]}, TypeError, "unsupported operand type(s) for +: 'float' and 'str'"),
    (P.call("sum", n("v"), "a"), {"v": [1]}, TypeError, "sum() can't sum strings [use ''.join(seq) instead]"),
    (P.call("sum", n("v"), b""), {"v": [1]}, TypeError, "sum() can't sum bytes [use b''.join(seq) instead]"),
    (P.call("sum", n("v")), {"v": [0.5, 10 ** 400]}, OverflowError, "int too large to convert to float"),
    (P.call("min", n("v")), {"v": []}, ValueError, "min() iterable argument is empty"),
    (P.call("max", P.generator("p", n("v"), n("p"))), {"v": []}, ValueError, "max() iterable argument is empty"),
    (P.call("max"), {}, TypeError, "max expected at least 1 argument, got 0"), (P.call("min", 1), {}, TypeError, "'int' object is not iterable"),
    (P.call("set", n("v")), {"v": [[1]]}, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"),
    (P.call("all", 5), {}, TypeError, "'int' object is not iterable"), (P.generator("p", 5, n("p")), {}, TypeError, "'int' object is not iterable"),
    (P.compare("in", 1, 5), {}, TypeError, "argument of type 'int' is not a container or iterable"),
    (P.compare("in", 1, "abc"), {}, TypeError, "'in <string>' requires string as left operand, not int"),
    (P.compare("in", "a", b"ab"), {}, TypeError, "a bytes-like object is required, not 'str'"),
    (P.compare("in", 256, b"ab"), {}, ValueError, "byte must be in range(0, 256)"),
    (P.compare("in", n("w"), n("v")), {"v": {1: 2}, "w": [1]}, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"),
    (P.compare("in", n("w"), P.call("set", n("v"))), {"v": [1], "w": [1]}, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"),
    (P.index(n("v"), 5), {"v": [1]}, IndexError, "list index out of range"), (P.index(n("v"), -3), {"v": [1, 2]}, IndexError, "list index out of range"),
    (P.index("abc", 5), {}, IndexError, "string index out of range"), (P.index(b"a", 3), {}, IndexError, "index out of range"),
    (P.index(n("v"), P.binop("**", 2, 70)), {"v": [1]}, IndexError, "cannot fit 'int' into an index-sized integer"),
    (P.index(n("v"), 1.5), {"v": [1]}, TypeError, "list indices must be integers or slices, not float"),
    (P.index("abc", 1.5), {}, TypeError, "string indices must be integers, not 'float'"),
    (P.index(b"ab", 1.5), {}, TypeError, "byte indices must be integers or slices, not float"),
    (P.index(5, 0), {}, TypeError, "'int' object is not subscriptable"), (P.index(n("v"), 5), {"v": {"k": 1}}, KeyError, "5"),
    (P.index(n("v"), n("w")), {"v": {"k": 1}, "w": ["k"]}, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"),
]
for expression, variables, error, message in ERRORS:
    with raises(error, match=message):
        PV.OfAny(expression, PV.Scope(variables, modules=modules))
assert PV.OfAny(P.importfrom("numpy.ma", "masked", n("masked")), PV.Scope(modules=modules)) is np.ma.masked

## DIA-03 · Matlab: operators, fields and `isfield`, rendered as MATLAB source

In [ ]:
s = M.identifier("s")
rule = M.binary("&&", M.binary(">=", M.field(s, "age"), 18), M.call("isfield", s, "email"))
assert MText.ToText(rule) == 's.age >= 18 && isfield(s, "email")'
assert MText.ToText(M.binary("-", 1, M.binary("-", 2, 3))) == "1 - (2 - 3)" and MText.ToText(
    M.binary("-", M.binary("-", 1, 2), 3)) == "1 - 2 - 3"
assert MText.ToText(M.binary(".*", M.binary("+", 1, 2), M.unary("-", M.unary("~", s)))) == "(1 + 2) .* -~s"
assert MText.ToText(M.unary("-", M.binary("+", 1, 2))) == "-(1 + 2)" and MText.ToText(M.unary("~", -1)) == "~-1"
assert MText.ToText(M.binary("||", M.binary("&&", True, False), M.binary("&&", 1, 0))) == "true && false || 1 && 0"
assert [MText.ToText(M.constant(v)) for v in [math.nan, math.inf, -math.inf, -2.5, 'say "hi"', 7]] == [
    "NaN", "Inf", "-Inf", "-2.5", '"say ""hi"""', "7"]
assert MText.ToText(M.field(M.field(s, "a"), "b")) == "s.a.b"
assert M.DIALECT.infer(rule, {"s": MD.Struct}) is MD.Logical
assert M.DIALECT.infer(M.binary("+", "a", 1)) is MD.String and M.DIALECT.infer(M.binary("+", True, 1)) is MD.Double
assert MD.Double.contains(1) and MD.Struct.contains({}) and not MD.Struct.contains("s") and MD.of(1) is MD.Double
assert M.binary("^", 1, 2).validate(core=True) == ["'^' is not a core operation"]

## DIA-04 · Matlab evaluates two-valued, converts numbers and logicals, and has no unknown

In [ ]:
assert MV.OfAny(rule, {"s": ann}) is True and MV.OfAny(rule, {"s": bob}) is False
assert MV.OfAny(rule, {"s": {"age": 20, "email": "x"}}) is True  # a mapping is a struct
assert MV.OfAny(M.field(s, "age"), {"s": ann}) == 30.0 and type(MV.OfAny(18)) is float  # numbers are doubles
assert MV.OfAny(M.binary("==", 1, True)) is True and MV.OfAny(M.binary("+", True, 1)) == 2.0
assert MV.OfAny(M.binary("<", "apple", "banana")) is True and MV.OfAny(M.binary("==", "5", 5)) is True
assert MV.OfAny(M.binary("==", "true", True)) is True and MV.OfAny(M.binary("~=", "1.5", 1.5)) is False
assert MV.OfAny(M.binary("+", "a", 1)) == "a1" and MV.OfAny(M.binary("+", "x", 0.125)) == "x0.125"
assert [MV.OfAny(M.binary("+", "x", v)) for v in [math.nan, math.inf, -math.inf, 12345.6, 1.5e-7, 123456.7, True,
                                                   False]] == [
    "xNaN", "xInf", "x-Inf", "x12346", "x1.5e-07", "x1.2346e+05", "xtrue", "xfalse"]  # numbers as MATLAB writes them
assert [MV.OfAny(M.binary(op, 2, 3)) for op in ["~=", "<=", ">", ">=", "-", ".*"]] == [True, True, False, False,
                                                                                        -1.0, 6.0]
assert MV.OfAny(M.binary("&&", False, M.identifier("missing"))) is False  # short-circuits
assert MV.OfAny(M.binary("||", 2, M.identifier("missing"))) is True and MV.OfAny(M.binary("||", 0, 0)) is False
assert MV.OfAny(M.unary("~", 0)) is True and MV.OfAny(M.unary("-", True)) == -1.0
assert MV.OfAny(M.call("isfield", 3, "x")) is False and MV.OfAny(M.call("isfield", s, 1), {"s": ann}) is False
with raises(KeyError, match='Unrecognized field name "email".'):
    MV.OfAny(M.field(s, "email"), {"s": bob})
with raises(TypeError, match="Dot indexing is not supported for variables of this type."):
    MV.OfAny(M.field(s, "age"), {"s": 3})
with raises(NameError, match="Unrecognized function or variable 'missing'."):
    MV.OfAny(M.identifier("missing"))
with raises(TypeError, match="Operands to the && operator must be convertible to logical scalar values."):
    MV.OfAny(M.binary("&&", "yes", True))
with raises(ValueError, match="NaN's cannot be converted to logicals."):
    MV.OfAny(M.unary("~", math.nan))
with raises(TypeError, match="Operator '-' is not supported for operands of type 'string' and 'double'."):
    MV.OfAny(M.binary("-", "a", 1))
with raises(TypeError, match="Operator '.*' is not supported for operands of type 'struct' and 'logical'."):
    MV.OfAny(M.binary(".*", s, True), {"s": ann})
with raises(TypeError, match="Operator '-' is not supported for operands of type 'string'."):
    MV.OfAny(M.unary("-", "a"))

## DIA-05 · Matlab resolves functions from imports, the path and packages that the scope provides

In [ ]:
geo = {"dist": lambda a, b: (a * a + b * b) ** 0.5, "area": lambda r: 3 * r * r}
scope = MV.Scope({"r": 2}, functions={"twice": lambda x: 2 * x}, packages={"geo": geo, "geo.more": {"area": lambda r: 0}})
wild = M.import_("geo.*", M.binary("+", M.call("dist", 3, 4), M.call("area", M.identifier("r"))))
assert MText.ToText(wild) == "import geo.*\ndist(3, 4) + area(r)" and wild.validate(bound={"r"}) == []
assert wild.validate(bound={"r"}, core=True) == ["body: left: 'dist' is not a core operation",
                                                 "body: right: 'area' is not a core operation"]
assert MV.OfAny(wild, scope) == 17.0  # an int result is a double
assert MV.OfAny(M.import_("geo.dist", M.call("dist", 6, 8)), scope) == 10.0
assert MV.OfAny(M.call("geo.more.area", 1), scope) == 0.0 and MV.OfAny(M.call("twice", 4), scope) == 8.0
assert MV.OfAny(M.import_("geo.*", M.import_("geo.more.area", M.call("area", 5))), scope) == 0.0  # innermost wins
with raises(NameError, match="Unrecognized function or variable 'dist'."):
    MV.OfAny(M.call("dist", 3, 4), scope)  # not imported
with raises(ImportError, match="Import argument 'os.system' cannot be found or cannot be imported."):
    MV.OfAny(M.import_("os.system", 1), scope)
with raises(ImportError, match="Import argument 'geo.nope' cannot be found or cannot be imported."):
    MV.OfAny(M.import_("geo.nope", 1), scope)
with raises(ImportError, match="Import argument 'nope.*' cannot be found or cannot be imported."):
    MV.OfAny(M.import_("nope.*", 1), scope)
with raises(ValueError, match="an import can only enclose the whole expression"):
    MText.ToText(M.binary("+", 1, M.import_("geo.*", 2)))
assert M.import_("geo", 1).validate() == ["an import's name must be pkg.name or pkg.*, got 'geo'"]
assert M.import_("geo.*.x", 1).validate() == ["an import's name must be pkg.name or pkg.*, got 'geo.*.x'"]
assert M.DIALECT.kinds()["import"](7, M.constant(1)).validate() == ["an import's name must be a str, got int"]

## DIA-06 · Excel: formulas with `LET`, functions, operators and fields

In [ ]:
r = X.name("r")
rule = X.let_("age", X.field(r, "age"), X.function("AND", X.infix(">=", X.name("age"), 18),
                                                     X.function("NOT", X.function("ISERROR", X.field(r, "email")))))
assert XText.ToText(rule) == "=LET(age, r.age, AND(age >= 18, NOT(ISERROR(r.email))))"
assert XText.ToText(X.infix("*", X.infix("+", 1, 2), X.prefix("-", X.infix("-", 3, 4)))) == "=(1 + 2) * -(3 - 4)"
assert XText.ToText(X.infix("=", X.infix("-", 1, X.infix("-", 2, 3)), -1)) == "=1 - (2 - 3) = -1"
assert XText.ToText(X.prefix("-", X.field(r, "first name"))) == "=-r.[first name]"
assert [XText.ToText(X.constant(v)) for v in [True, False, 'say "hi"', math.inf, 2.5, 3]] == [
    "=TRUE", "=FALSE", '="say ""hi"""', "=#NUM!", "=2.5", "=3"]
assert rule.validate(bound={"r"}, core=True) == [] and X.function("VLOOKUP", 1, 2).validate(core=True) == [
    "'VLOOKUP' is not a core operation"]
assert X.DIALECT.infer(rule, {"r": XD.Record}) is XD.Logical and X.DIALECT.infer(X.infix("+", "1", True)) is XD.Number
assert XD.of(1) is XD.Number and XD.Errors.contains(XD.Error("#N/A")) and repr(XD.Error("#N/A")) == "#N/A"
assert XD.Record.contains(ann) and not XD.Record.contains(1)

## DIA-07 · Excel coerces, orders values of every type, and computes with errors

In [ ]:
Error = XV.Error
assert XV.OfAny(rule, {"r": ann}) is True and XV.OfAny(rule, {"r": bob}) is False  # ISERROR finds the #FIELD!
assert XV.OfAny(X.field(r, "email"), {"r": bob}) == Error("#FIELD!") and XV.OfAny(X.name("nope")) == Error("#NAME?")
assert XV.OfAny(X.field(r, "age"), {"r": {"age": 5}}) == 5.0 and XV.OfAny(X.field(r, "x"), {"r": 3}) == Error("#VALUE!")
assert XV.OfAny(X.field(X.field(r, "a"), "b"), {"r": {}}) == Error("#FIELD!")  # errors propagate
assert XV.OfAny(X.infix("+", "1", True)) == 2.0 and XV.OfAny(X.infix("*", "x", 1)) == Error("#VALUE!")  # coercion
assert XV.OfAny(X.infix("-", 1, X.infix("+", X.name("u"), 1))) == Error("#NAME?")
assert XV.OfAny(X.prefix("-", "2")) == -2.0 and XV.OfAny(X.prefix("-", X.name("u"))) == Error("#NAME?")
assert XV.OfAny(X.infix(">", "a", 1)) is True and XV.OfAny(X.infix(">", True, "z")) is True  # number < text < logical
assert XV.OfAny(X.infix("=", "a", "A")) is True and XV.OfAny(X.infix("<>", 1, "1")) is True  # case; no coercion
assert [XV.OfAny(X.infix(op, 2, 3)) for op in ["<", "<=", ">=", "-"]] == [True, True, False, -1.0]
assert XV.OfAny(X.infix("=", 2, 2)) is True and XV.OfAny(X.infix("+", False, 1)) == 1.0
assert XV.OfAny(X.infix("+", "inf", 1)) == Error("#VALUE!") and XV.OfAny(X.infix("+", " 1e3 ", 1)) == 1001.0  # decimal text
assert XV.OfAny(X.infix("+", r, 1), {"r": {}}) == Error("#VALUE!")
assert XV.OfAny(X.infix("=", X.name("u"), 1)) == Error("#NAME?") and XV.OfAny(X.infix("=", r, 1), {"r": {}}) == Error(
    "#VALUE!")
assert XV.OfAny(X.function("AND", 1, True)) is True and XV.OfAny(X.function("OR", 0, False)) is False
assert XV.OfAny(X.function("AND", False, X.name("u"))) == Error("#NAME?")  # AND evaluates every argument
assert XV.OfAny(X.function("OR", "yes", True)) == Error("#VALUE!") and XV.OfAny(X.function("NOT", 0)) is True
assert XV.OfAny(X.function("NOT", X.name("u"))) == Error("#NAME?")
assert XV.OfAny(X.function("IF", True, 1, X.name("u"))) == 1.0 and XV.OfAny(X.function("IF", 0, 1, "no")) == "no"
assert XV.OfAny(X.function("IF", X.name("u"), 1, 2)) == Error("#NAME?")  # IF evaluates only the branch it takes
assert XV.OfAny(X.function("ISERROR", 1)) is False and XV.OfAny(X.let_("x", 2, X.infix("*", X.name("x"), 3))) == 6.0

## DIA-08 · Excel reads cells from workbooks, and add-in functions from the workbook's add-ins

In [ ]:
rates = XV.Workbook(sheets={"Rates": {"B2": 0.5}}, name="Rates.xlsx")
book = XV.Workbook({"base": 10}, {"Sheet1": {"a1": 2, "$A$2": 3}, "My Sheet": {"C3": "x"}}, books={"Rates.xlsx": rates},
                   add_ins={"double": lambda v: v * 2})
total = X.infix("+", X.infix("*", X.cell("A1"), X.cell("B2", "Rates", "Rates.xlsx")), X.function("DOUBLE", X.name("base")))
assert XText.ToText(total) == "=A1 * [Rates.xlsx]Rates!B2 + DOUBLE(base)" and XV.OfAny(total, book) == 21.0
assert total.validate(bound={"base"}) == [] and total.validate(bound={"base"}, core=True) == [
    "right: 'DOUBLE' is not a core operation"]
assert XText.ToText(X.cell("C3", "My Sheet")) == "='My Sheet'!C3" and XV.OfAny(X.cell("C3", "My Sheet"), book) == "x"
assert XText.ToText(X.cell("A1", "Bob's", "My Book.xlsx")) == "='[My Book.xlsx]Bob''s'!A1"
assert XV.OfAny(X.cell("$a$2"), book) == 3.0 and XV.OfAny(X.cell("Z9"), book) == 0.0  # an empty cell is 0
assert XV.OfAny(X.cell("A1", "Sheet1", "Book1"), book) == 2.0  # this book, named
assert XV.OfAny(X.cell("A1", "Nope"), book) == Error("#REF!") and XV.OfAny(X.cell("A1", "S", "Other.xlsx"), book) == Error(
    "#REF!")
assert XV.OfAny(X.function("NOPE", 1), book) == Error("#NAME?") and XV.OfAny(X.function("double", X.name("u")), book) == Error(
    "#NAME?")  # add-ins ignore case; errors in their arguments propagate
assert XV.OfAny(X.cell("A1"), XV.Workbook()) == Error("#REF!")  # a workbook has no sheet until given one
assert XV.OfAny(X.name("x"), {"x": 1}) == 1.0 and XV.Workbook().sheet == "Sheet1"
with raises(ValueError, match="not a cell address: 'total'"):
    XV.Workbook(sheets={"Sheet1": {"total": 1}})
assert X.cell("A0").validate() == ["a cell's address must be a column and a row, like A1, got 'A0'"]
assert X.cell("A1", None, "B.xlsx").validate() == ["a cell in another book needs a sheet"]
assert X.DIALECT.infer(X.cell("A1")) is D.Anything and X.address("$b$12") == "B12" and X.address("B") is None

## DIA-09 · Latex: notation in math mode, rendered and checked, with no evaluator

In [ ]:
a, this = L.symbol("a"), L.symbol("this")
notation = L.where("a", L.member(this, "age"), L.binary("\\land", L.binary("\\geq", a, 18), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", this, "email")), L.binary(">", L.frac(a, 2), 1.5))))
assert LText.ToText(notation) == (
    "a \\geq 18 \\land (\\lnot \\operatorname{has}(\\mathit{this}, \\text{email}) \\lor \\frac{a}{2} > 1.5) "
    "\\quad \\text{where } a = \\mathit{this}.\\mathit{age}")
assert LText.ToText(L.binary("\\cdot", L.binary("+", 1, 2), L.unary("-", L.binary("-", 3, -4)))) == "(1 + 2) \\cdot -(3 - -4)"
assert LText.ToText(L.binary("\\implies", L.binary("\\implies", 1, 2), L.binary("\\implies", 3, 4))) == (
    "(1 \\implies 2) \\implies 3 \\implies 4")  # right-associative
assert LText.ToText(L.binary("<", L.binary("<", 1, 2), 3)) == "(1 < 2) < 3"  # comparisons do not chain
assert LText.ToText(L.member(L.binary("+", this, 1), "x_y")) == "(\\mathit{this} + 1).\\mathit{x\\_y}"
assert LText.ToText(L.where("b", 1, L.where("a", 2, a))) == (
    "(a \\quad \\text{where } a = 2) \\quad \\text{where } b = 1")
assert [LText.ToText(L.constant(v)) for v in [math.nan, math.inf, -math.inf, 1e-7, 1e16, -2, True, False, "a_b {c}"]] == [
    "\\mathrm{NaN}", "\\infty", "-\\infty", "1 \\times 10^{-7}", "1 \\times 10^{16}", "-2", "\\mathrm{true}",
    "\\mathrm{false}", "\\text{a\\_b \\{c\\}}"]
assert LText.ToText(L.constant("\\$&#%~^")) == "\\text{\\textbackslash{}\\$\\&\\#\\%\\textasciitilde{}\\textasciicircum{}}"
assert LText.ToText(L.binary("\\cdot", L.constant(1e16), 2)) == "1 \\times 10^{16} \\cdot 2"
assert notation.validate(bound={"this"}, core=True) == [] and L.binary("\\times", 1, 2).validate(core=True) == [
    "'\\\\times' is not a core operation"]
assert L.function("max", 1, 2).validate() == [] and L.function("has", 1).validate() == ["has takes 2 arguments, got 1"]
assert L.DIALECT.infer(notation, {"this": D.Anything}) is LD.Truth and L.DIALECT.infer(L.frac(1, 2.5)) is LD.Number
with raises(TypeError, match="\\land cannot take (number, truth)"):
    L.DIALECT.infer(L.binary("\\land", 1, True))
assert LD.of("x") is LD.Text and not hasattr(LatexPackage, "Evaluators")  # notation is not evaluated

## DIA-10 · Bitwise operations: Python's operators, and MATLAB's and Excel's bit functions

In [ ]:
# Python: rendered and parsed with Python's precedence, and evaluated by its rules (DIA-02's tables have more).
for source in ["a | b ^ c & d << 1 >> 2", "(a | b) & c", "~x + 1", "(a << b) + c", "x & 1 == 1", "-~x"]:
    assert PText.ToText(PText.FromText(source)) == source
assert PV.OfAny(PText.FromText("(x & 12 | x >> 2) ^ x << 1"), {"x": 13}) == 21
assert P.DIALECT.infer(PText.FromText("True & False")) is PD.Bool and P.DIALECT.infer(PText.FromText("1 << True")) is PD.Int
assert P.DIALECT.infer(PText.FromText("~x"), {"x": PD.Bool}) is PD.Int
# MATLAB: the bit functions take doubles holding integers of 53 bits.
assert MV.OfAny(M.call("bitand", 12, 10)) == 8.0 and MV.OfAny(M.call("bitor", 12, True)) == 13.0
assert MV.OfAny(M.call("bitxor", 12, 10)) == 6.0 and MV.OfAny(M.call("bitshift", 1, 3)) == 8.0
assert MV.OfAny(M.call("bitshift", 16, M.unary("-", 2))) == 4.0 and MV.OfAny(M.call("bitshift", 2.0 ** 52, 1)) == 0.0  # beyond 53 bits
assert MText.ToText(M.call("bitshift", M.identifier("x"), M.unary("-", 2))) == "bitshift(x, -2)"
assert M.DIALECT.infer(M.call("bitand", 1, True)) is MD.Double
for call in [M.call("bitand", -1, 1), M.call("bitand", 1.5, 1), M.call("bitor", 2.0 ** 54, 1), M.call("bitshift", 1, 0.5)]:
    with raises(ValueError, match="Double inputs must have integer values in the range of ASSUMEDTYPE."):
        MV.OfAny(call)
with raises(TypeError, match="Undefined function 'bitor' for input arguments of type 'string'."):
    MV.OfAny(M.call("bitor", "a", 1))
# Excel: the bit functions take integers below 2^48, converted as arithmetic converts; otherwise #NUM!.
assert XV.OfAny(X.function("BITAND", 12, 10)) == 8.0 and XV.OfAny(X.function("BITOR", "12", True)) == 13.0
assert XV.OfAny(X.function("BITXOR", 12, 10)) == 6.0 and XV.OfAny(X.function("BITLSHIFT", 1, 3)) == 8.0
assert XV.OfAny(X.function("BITRSHIFT", 16, 2)) == 4.0 and XV.OfAny(X.function("BITRSHIFT", 1, -3)) == 8.0
assert XV.OfAny(X.function("BITLSHIFT", 16, -2)) == 4.0 and XText.ToText(X.function("BITXOR", X.name("x"), 1)) == "=BITXOR(x, 1)"
for formula in [X.function("BITAND", -1, 1), X.function("BITAND", 1.5, 1), X.function("BITOR", 2 ** 48, 1),
                X.function("BITLSHIFT", 1, 54), X.function("BITLSHIFT", 1, 48), X.function("BITRSHIFT", 1, 0.5)]:
    assert XV.OfAny(formula) == XV.Error("#NUM!"), XText.ToText(formula)
assert XV.OfAny(X.function("BITAND", "a", 1)) == XV.Error("#VALUE!")
assert XV.OfAny(X.function("BITXOR", 1, X.name("nope"))) == XV.Error("#NAME?")
assert X.DIALECT.infer(X.function("BITAND", 1, 2)) is XD.Number

## DIA-11 · Ccpp: C and C++ expressions over their arithmetic types, with C's precedence, typed constants and casts

In [ ]:
x, y, me = C.identifier("x"), C.identifier("y"), C.identifier("this")
cases = [
    (C.binary("+", x, C.binary("*", y, 2)), "x + y * 2"), (C.binary("*", C.binary("+", x, y), 2), "(x + y) * 2"),
    (C.binary("-", x, C.binary("-", y, 1)), "x - (y - 1)"), (C.binary("<<", C.binary("+", x, 1), 2), "x + 1 << 2"),
    (C.binary("&", C.binary("==", x, 1), y), "x == 1 & y"), (C.binary("==", C.binary("&", x, 1), y), "(x & 1) == y"),
    (C.binary("||", C.binary("&&", x, y), C.unary("!", x)), "x && y || !x"),
    (C.binary("|", C.binary("^", x, y), C.binary("%", x, y)), "x ^ y | x % y"),
    (C.binary("<", C.binary(">>", x, 1), C.binary("/", y, 2)), "x >> 1 < y / 2"),
    (C.conditional(x, y, C.conditional(y, 1, 2)), "x ? y : y ? 1 : 2"), (C.conditional(C.conditional(x, y, 1), 2, 3), "(x ? y : 1) ? 2 : 3"),
    (C.unary("-", C.unary("-", x)), "- -x"), (C.unary("-", -1), "- -1"), (C.unary("~", C.unary("!", x)), "~!x"),
    (C.cast("uint8_t", C.binary("+", x, 1)), "(uint8_t)(x + 1)"), (C.unary("-", C.cast("int", x)), "-(int)x"),
    (C.member(C.member(me, "a"), "b", "->"), "this.a->b"), (C.subscript(C.member(me, "ports"), C.binary("+", x, 1)), "this.ports[x + 1]"),
    (C.call("f", x, C.binary("+", y, 1)), "f(x, y + 1)"), (C.member(C.binary("+", x, y), "a"), "(x + y).a"),
    (C.constant(5, "unsigned int"), "5u"), (C.constant(5, "unsigned long"), "5ul"), (C.constant(5, "long long"), "5ll"),
    (C.constant(5, "unsigned long long"), "5ull"), (C.constant(5, "long"), "5l"), (C.constant(5, "int"), "5"),
    (C.constant(1.5, "float"), "1.5f"), (C.constant("1.5", "long double"), "1.5L"), (C.constant("1", "long double"), "1.0L"),
    (C.constant("Infinity", "long double"), "INFINITY"), (C.constant(5, "uint8_t"), "(uint8_t)5"), (C.constant(2.5, "double"), "2.5"),
    (C.constant(1.0), "1.0"), (C.constant(1e300), "1e+300"), (C.constant(math.nan), "NAN"), (C.constant(-math.inf), "-INFINITY"), (C.constant(math.inf), "INFINITY"),
    (C.constant(True), "true"), (C.constant(False), "false"), (C.constant('a"\\\n\t\r'), '"a\\"\\\\\\n\\t\\r"'),
    (C.constant(-5), "-5"), (C.binary("-", x, -5), "x - -5"),
]
for expression, source in cases:
    assert CText.ToText(expression) == source, (source, CText.ToText(expression))
# Validation: types are C's arithmetic types, and hold the constant's value.
assert C.constant(5, "uint9_t").validate() == ["a constant's type must be one of C's arithmetic types, got 'uint9_t'"]
assert C.constant(300, "uint8_t").validate() == ["a constant of uint8_t cannot hold 300"]
assert C.cast("word", x).validate(bound={"x"}) == ["a cast's type must be one of C's arithmetic types, got 'word'"]
assert C.member(x, "a", "=>").validate(bound={"x"}) == ["a member's operator must be '.' or '->', got '=>'"]
assert C.binary("**", x, 1).validate(bound={"x"}, core=True) == ["'**' is not a core operation"]
assert C.call("f", x).validate(bound={"x"}, core=True) == []  # functions are the scope's
assert C.DIALECT.kinds()["cast"]().validate() == ["a cast needs a type", "a cast needs an operand"]
# Types under LP64, the integer promotions and the usual arithmetic conversions.
T = CD.TYPES
assert T["int32_t"] == T["int"] and T["int32_t"].name() == "int32_t" and repr(T["size_t"]) == "size_t" and T["size_t"] == T["unsigned long"]
assert CD.promoted(T["uint8_t"]) is CD.Int and CD.promoted(CD.Bool) is CD.Int and CD.promoted(T["long"]) is T["long"]
assert CD.common(CD.Int, T["unsigned int"]).name() == "unsigned int" and CD.common(T["long"], T["unsigned int"]).name() == "long"
assert CD.common(T["float"], T["long"]).name() == "float" and CD.common(CD.Double, T["long double"]).name() == "long double"
assert CD.common(T["unsigned long"], T["long long"]) == T["unsigned long"] and CD.common(T["short"], T["char"]) is CD.Int
assert CD.common(CD.Int, T["long long"]).name() == "long long" and CD.common(T["long"], CD.Int).name() == "long"
assert [CD.of(v) for v in [1, 2 ** 40, 2 ** 63, True, 1.5, "a"]] == [CD.Int, T["long"], T["unsigned long"], CD.Bool, CD.Double, CD.String]
assert CD.of(2 ** 64) is D.Anything and CD.of(None) is D.Anything
assert T["uint8_t"].contains(255) and not T["uint8_t"].contains(256) and T["int8_t"].contains(-128) and not T["int8_t"].contains(-129)
assert T["float"].contains(1.5) and T["long double"].contains("1.5") and CD.Bool.contains(True) and not CD.Int.contains(True)
assert not CD.Int.contains(1.5) and CD.Int.includes(T["int32_t"]) and CD.Int.includes(D.OfUnion(CD.Int)) and not CD.Int.includes(T["long"])
infer = C.DIALECT.infer
assert infer(C.binary("+", C.constant(1, "uint8_t"), 1)) is CD.Int and infer(C.binary("+", 1, C.constant(1, "unsigned int"))).name() == "unsigned int"
assert infer(C.binary("<<", C.constant(1, "uint8_t"), C.constant(1, "long"))) is CD.Int and infer(C.binary("<", 1, 2.0)) is CD.Bool
assert infer(C.binary("&&", x, 1), {"x": D.Anything}) is CD.Bool and infer(C.binary("+", x, 1), {"x": D.Anything}) is D.Anything
assert infer(C.cast("uint16_t", x), {"x": D.Anything}) == T["uint16_t"] and infer(C.constant(5, "uint8_t")) == T["uint8_t"]
assert infer(C.unary("~", C.constant(1, "uint8_t"))) is CD.Int and infer(C.unary("-", 1.5)) is CD.Double and infer(C.unary("!", 1.5)) is CD.Bool
assert infer(C.binary("%", C.constant(7, "uint32_t"), C.constant(2, "uint32_t"))).name() == "unsigned int"
assert infer(C.conditional(x, 1, 2), {"x": D.Anything}) is D.Anything and infer(C.member(x, "a"), {"x": D.Anything}) is D.Anything
for expression, message in [(C.binary("%", 1.5, 1), "% cannot take (double, int); it takes (integer, integer) -> their common type"),
                            (C.binary("+", "a", 1), "+ cannot take (string, int)"), (C.binary("<<", 1.5, 1), "<< cannot take (double, int)"),
                            (C.unary("~", 1.5), "~ cannot take (double); it takes (integer) -> its promoted type")]:
    with raises(TypeError, match=message):
        infer(expression)

## DIA-12 · Ccpp evaluates by C's rules: promotions and conversions, wrapping unsigned arithmetic, and undefined behavior raising

In [ ]:
def ev(expression, **variables):
    return CV.OfAny(expression, variables)


x = C.identifier("x")
U = lambda value: C.constant(value, "unsigned int")  # noqa: E731
assert ev(C.binary("+", U(4294967295), 1)) == 0 and ev(C.binary("-", U(0), 1)) == 4294967295 and ev(C.binary("*", U(65536), U(65536))) == 0
assert ev(C.binary("<", -1, U(0))) is False and ev(C.binary("<", -1, C.constant(0, "unsigned short"))) is True  # promoted to int
assert ev(C.binary("+", C.constant(200, "uint8_t"), C.constant(100, "uint8_t"))) == 300  # in int
assert ev(C.binary("/", -7, 2)) == -3 and ev(C.binary("%", -7, 2)) == -1 and ev(C.binary("/", 7, -2)) == -3 and ev(C.binary("%", 7, -2)) == 1
assert ev(C.binary("/", 7.0, 2)) == 3.5 and ev(C.binary("/", 1.0, 0.0)) == math.inf and math.isnan(ev(C.binary("/", 0.0, 0.0)))
assert ev(C.binary("+", C.cast("float", 1.5), C.cast("float", 0.1))) == 1.600000023841858 and ev(C.binary("-", 1.5, 1)) == 0.5
assert ev(C.binary("*", C.cast("long double", 1.5), 2)) == "3"
assert ev(C.binary("&", 12, 10)) == 8 and ev(C.binary("^", 12, 10)) == 6 and ev(C.binary("|", -1, U(0))) == 4294967295
assert ev(C.binary("<<", U(1), 31)) == 2147483648 and ev(C.binary(">>", -8, 1)) == -4 and ev(C.binary("<<", C.constant(1, "uint8_t"), 8)) == 256
assert ev(C.unary("-", U(1))) == 4294967295 and ev(C.unary("~", C.constant(0, "uint8_t"))) == -1 and ev(C.unary("~", U(0))) == 4294967295
assert ev(C.unary("+", True)) == 1 and ev(C.unary("-", 1.5)) == -1.5 and ev(C.unary("+", 1.5)) == 1.5
assert ev(C.unary("!", 0.0)) is True and ev(C.unary("!", -0.0)) is True and ev(C.unary("!", math.nan)) is False and ev(C.unary("!", 3)) is False
assert ev(C.binary("==", math.nan, math.nan)) is False and ev(C.binary("!=", math.nan, 1.0)) is True and ev(C.binary("==", -0.0, 0.0)) is True
assert ev(C.binary("<", 1, 1.5)) is True and ev(C.binary(">=", C.cast("long double", 2), 1)) is True and ev(C.binary(">", 2, 1)) is True
assert ev(C.binary("<=", math.inf, math.inf)) is True and ev(C.binary("<", C.cast("long double", math.nan), 1)) is False
assert ev(C.binary("&&", 0, C.binary("/", 1, 0))) is False and ev(C.binary("||", 2, C.binary("/", 1, 0))) is True  # decided early
assert ev(C.binary("&&", 1, 2.5)) is True and ev(C.binary("||", 0, 0.0)) is False
assert ev(C.conditional(C.binary(">", x, 0), C.binary("/", 10, x), 0), x=0) == 0 and ev(C.conditional(1, 2, 3.5)) == 2  # the chosen one
assert ev(C.cast("uint8_t", 300)) == 44 and ev(C.cast("int8_t", 200)) == -56 and ev(C.cast("int", 2.9)) == 2 and ev(C.cast("int", -2.9)) == -2
assert ev(C.cast("bool", 0.5)) is True and ev(C.cast("bool", 0)) is False and ev(C.cast("bool", -0.0)) is False
assert ev(C.cast("double", 2 ** 60 + 1)) == 2.0 ** 60 and ev(C.cast("float", 0.1)) == 0.10000000149011612 and ev(C.cast("int", True)) == 1
assert ev(C.cast("long double", 0.1)) == "0.1000000000000000055511151231257827" and ev(C.cast("double", C.cast("float", 0.1))) == 0.10000000149011612
assert ev(C.constant(5, "uint8_t")) == 5 and ev(C.constant("hi")) == "hi"
# What C leaves undefined raises.
for expression, error, message in [
    (C.binary("+", 2147483647, 1), OverflowError, "signed overflow in +: 2147483648 does not fit 'int'"),
    (C.unary("-", -2147483648), OverflowError, "signed overflow in -: 2147483648 does not fit 'int'"),
    (C.binary("/", -2147483648, -1), OverflowError, "signed overflow in /: 2147483648 does not fit 'int'"),
    (C.binary("%", 1, 0), ZeroDivisionError, "integer division by zero"),
    (C.binary("<<", 1, 32), ValueError, "shift by 32 is undefined for 'int'"), (C.binary(">>", 1, -1), ValueError, "shift by -1 is undefined for 'int'"),
    (C.binary("<<", -1, 1), ValueError, "left shift of a negative value is undefined"),
    (C.binary("<<", 1, 31), OverflowError, "signed overflow in <<: 2147483648 does not fit 'int'"),
    (C.cast("uint8_t", 300.0), OverflowError, "300.0 is out of range of 'uint8_t'"),
    (C.cast("int", math.inf), OverflowError, "inf is out of range of 'int'"),
    (C.cast("int", C.cast("long double", 1e300)), OverflowError, " is out of range of 'int'"),
    (C.cast("int", math.nan), ValueError, "NaN has no integer value"),
    (C.binary("%", 1.5, 1), TypeError, "invalid operands to binary % ('double' and 'double')"),
    (C.binary("<<", 1.5, 1), TypeError, "invalid operands to binary << ('double' and 'int')"),
    (C.unary("~", 1.5), TypeError, "invalid argument type 'double' to unary expression"),
    (C.binary("+", "a", 1), TypeError, "invalid operand to +: 'str'"), (C.unary("!", "a"), TypeError, "invalid operand to !: 'str'"),
]:
    with raises(error, match=message):
        ev(expression)
# A variable's type is its value's: a typed value's by its width and signedness.
with raises(OverflowError, match="integer 18446744073709551616 is too large for any type"):
    ev(x, x=2 ** 64)
with raises(TypeError, match="enum(A) has no C type"):
    ev(x, x=BD.Value(BD.OfEnum.Data(("A",)), "A"))
U8, F32 = BD.OfInteger.Data(8, False), BD.OfIeee754.Data("binary32")
assert ev(C.binary("+", x, 1), x=BD.Value(U8, 255)) == 256 and ev(C.cast("uint8_t", C.binary("+", x, 1)), x=BD.Value(U8, 255)) == 0
assert ev(C.binary("+", x, x), x=BD.Value(F32, 1.5)) == 3.0 and ev(C.binary("+", x, 1), x=2 ** 40) == 2 ** 40 + 1
assert ev(C.binary("*", x, 2), x=BD.Value(BD.OfInteger.Data(16), -5)) == -10 and ev(C.constant(5, "word")) == 5  # no type: its value's
# Members, elements and calls.
Board = S.OfObject.Builder().name("Board").ref().properties(text("name"), lambda p: p.name("grid").of(
    lambda t: t.as_indexed(lambda i: i.of(lambda t: t.as_indexed(lambda j: j.of(S.OfNative.Data(int))))))).create()
store.register(Board)
board = store.Board().name("b").grid([[1, 2], [3]]).create()
assert ev(C.member(x, "age"), x=ann) == 30 and ev(C.member(x, "age", "->"), x={"age": 5}) == 5 and ev(C.member(x, "age"), x={"age": 7}) == 7
assert ev(C.subscript(x, 1), x=[10, 20]) == 20 and ev(C.subscript(C.subscript(C.member(x, "grid"), 1), 0), x=board) == 3
for expression, variables, error, message in [
    (C.member(x, "phone"), ann, KeyError, "no member named 'phone'"),
    (C.member(x, "a"), 1, TypeError, "member reference base type 'int' is not a structure"),
    (C.subscript(x, 2), [10, 20], ValueError, "index 2 is out of bounds of an array of 2"),
    (C.subscript(x, 0), 1, TypeError, "subscripted value is not an array: 'int'"),
    (C.subscript(x, 0.5), [1], TypeError, "array subscript is not an integer: 'double'"),
    (C.call("f", 1), None, FE.NameError, "use of undeclared identifier 'f'"), (C.identifier("z"), None, FE.NameError, "use of undeclared identifier 'z'"),
]:
    with raises(error, match=message):
        ev(expression, x=variables)
assert CV.OfAny(C.call("max", x, 3), CV.Scope({"x": 5}, functions={"max": max})) == 5

## DIA-13 · SystemVerilog: expressions over 4-state vectors and reals, with IEEE 1800's precedence, sized literals and casts

In [ ]:
x, y, v = SV.identifier("x"), SV.identifier("y"), SV.vector
cases = [
    (SV.binary("+", x, SV.binary("*", y, 2)), "x + y * 2"), (SV.binary("*", SV.binary("+", x, y), 2), "(x + y) * 2"),
    (SV.binary("-", x, SV.binary("-", y, 1)), "x - (y - 1)"), (SV.binary("**", SV.binary("**", x, y), 2), "x ** y ** 2"),
    (SV.binary("**", x, SV.binary("**", y, 2)), "x ** (y ** 2)"), (SV.binary("<<", SV.binary("+", x, 1), 2), "x + 1 << 2"),
    (SV.binary("&&", SV.binary("||", x, y), x), "(x || y) && x"), (SV.binary("||", SV.binary("&&", x, y), x), "x && y || x"),
    (SV.binary("^~", SV.binary("|", x, y), SV.binary("&", x, y)), "(x | y) ^~ x & y"),
    (SV.binary("==", SV.binary("&", x, 1), y), "(x & 1) == y"), (SV.binary("<", SV.binary(">>>", x, 1), SV.binary("%", y, 2)), "x >>> 1 < y % 2"),
    (SV.binary("->", SV.binary("->", x, y), x), "(x -> y) -> x"), (SV.binary("<->", x, SV.binary("->", y, x)), "x <-> y -> x"),
    (SV.binary("->", SV.conditional(x, y, 1), y), "x ? y : 1 -> y"), (SV.conditional(SV.binary("->", x, y), 1, 2), "(x -> y) ? 1 : 2"),
    (SV.conditional(x, y, SV.conditional(y, 1, 2)), "x ? y : y ? 1 : 2"), (SV.conditional(SV.conditional(x, y, 1), 2, 3), "(x ? y : 1) ? 2 : 3"),
    (SV.unary("-", SV.unary("-", x)), "- -x"), (SV.unary("&", SV.unary("&", x)), "& &x"), (SV.unary("~", SV.unary("!", x)), "~ !x"),
    (SV.unary("-", -1), "- -1"), (SV.binary("-", x, -1), "x - -1"), (SV.unary("~&", SV.binary("+", x, 1)), "~&(x + 1)"),
    (SV.inside(SV.binary("+", x, 1), 1, SV.span(2, 5)), "x + 1 inside {1, [2:5]}"), (SV.binary("<", SV.inside(x, 1), 0), "x inside {1} < 0"),
    (SV.binary("+", SV.inside(x, 1), 1), "(x inside {1}) + 1"), (SV.binary("==?", x, v("1x")), "x ==? 2'b1x"),
    (v("11111111", base="h"), "8'hff"), (v("10x1", base="h"), "4'b10x1"), (v("1xxxx", base="h"), "5'h1x"), (v("1zzz", base="h"), "4'b1zzz"),
    (v("00000101", True, "d"), "8'sd5"), (v("xxxx", base="d"), "4'dx"), (v("0x", base="d"), "2'b0x"), (v("111", base="o"), "3'o7"),
    (v("10z1"), "4'b10z1"), (v("1", True), "1'sb1"),
    (SV.constant(1.5), "1.5"), (SV.constant(1.0), "1.0"), (SV.constant(1e300), "1e+300"), (SV.constant(math.nan), "0.0 / 0.0"),
    (SV.constant(math.inf), "1.0 / 0.0"), (SV.constant(-math.inf), "-1.0 / 0.0"), (SV.binary("*", x, math.inf), "x * (1.0 / 0.0)"),
    (SV.unary("-", math.inf), "-(1.0 / 0.0)"), (SV.constant('a"\\\n\t'), '"a\\"\\\\\\n\\t"'), (SV.constant(-5), "-5"),
    (SV.cast("signed", x), "signed'(x)"), (SV.cast(8, SV.binary("+", x, 1)), "8'(x + 1)"), (SV.cast("byte", -1), "byte'(-1)"),
    (SV.member(SV.member(x, "a"), "b"), "x.a.b"), (SV.select(SV.binary("+", x, y), 0), "(x + y)[0]"), (SV.range_(x, 7, 4), "x[7:4]"),
    (SV.concatenation(x, SV.replication(3, y)), "{x, {3{y}}}"), (SV.call("$clog2", x), "$clog2(x)"), (SV.call("f", x, 1), "f(x, 1)"),
]
for expression, source in cases:
    assert SVText.ToText(expression) == source, (source, SVText.ToText(expression))
# Validation: a vector's bits and base, and a cast's type or width.
assert v("12").validate() == ["a vector's bits must be 0, 1, x and z, at least one, got '12'"]
assert v("").validate() == ["a vector's bits must be 0, 1, x and z, at least one, got ''"]
assert v("1", base="q").validate() == ["a vector's base must be b, o, d or h, got 'q'"]
assert SV.cast("word", x).validate(bound={"x"}) == ["a cast's type must be a built-in type, signed or unsigned, got 'word'"]
assert SV.cast(0, x).validate(bound={"x"}) == ["a cast's width must be positive, got 0"]
assert SV.DIALECT.kinds()["cast"](None, None, x).validate(bound={"x"}) == ["a cast names a type or a width, not both nor neither"]
assert SV.binary("<<=", x, 1).validate(bound={"x"}, core=True) == ["'<<=' is not a core operation"]
assert SV.call("f", x).validate(bound={"x"}, core=True) == [] and SV.identifier("x").validate() == ["identifier 'x' is not bound"]
# Types: integral vectors of a width, a signedness and 2 or 4 states, and reals.
T = SVD.TYPES
assert [T[n].width for n in ["bit", "logic", "reg", "byte", "shortint", "int", "longint", "integer", "time"]] == [1, 1, 1, 8, 16, 32, 64, 32, 64]
assert T["int"].signed and T["int"].states == 2 and T["integer"].states == 4 and not T["time"].signed and T["logic"] == T["reg"]
assert T["int"].name() == "int" and repr(T["integer"]) == "integer" and SVD.vector(8).name() == "logic [7:0]"
assert SVD.vector(4, True, 2).name() == "bit signed [3:0]" and SVD.vector(32, True, 2) == T["int"] and T["shortreal"].format == "binary32"
assert SVD.common([T["byte"], T["int"]]) == T["int"] and SVD.common([T["byte"], SVD.vector(16)]) == SVD.vector(16)
assert SVD.common([T["shortreal"], T["int"]]) == T["real"] and SVD.common([T["shortreal"], T["shortreal"]]) == T["shortreal"]
assert SVD.common([T["bit"], T["logic"]]) == T["logic"]
assert [SVD.of(value) for value in [1, -2 ** 31, 2 ** 31, 2 ** 63, 1.5, "a", True]] == [T["integer"], T["integer"], T["longint"], D.Anything,
                                                                                         T["real"], D.Anything, D.Anything]
# Values: a Logic is a type and its bits, encoded as VPI does.
a = SVD.Logic.of("10xz")
assert (a.aval, a.bval, a.bits, a.known, repr(a)) == (0b1010, 0b0011, "10xz", False, "4'b10xz")
assert SVD.Logic.of("10xz", True, 2).bits == "1000" and repr(SVD.Logic.of("1", True)) == "1'sb1"
assert SVD.Logic.of("1110", True).integer() == -2 and SVD.Logic.of("1110").integer() == 14 and SVD.Logic.number(T["byte"], -1).bits == "11111111"
with raises(ValueError, match="4'b10xz has unknown bits"):
    a.integer()
assert T["int"].contains(SVD.Logic.number(T["int"], 5)) and not T["int"].contains(SVD.Logic.number(T["integer"], 5)) and not T["int"].contains(5)
assert T["real"].contains(1.5) and not T["real"].contains(1) and T["int"].includes(SVD.vector(32, True, 2)) and not T["int"].includes(T["integer"])
assert T["int"].includes(D.OfUnion(T["int"])) and not T["int"].includes(D.Anything)
# Inference sizes an expression as IEEE 1800 does.
infer = SV.DIALECT.infer
assert infer(SV.binary("+", v("1010"), 1)) == SVD.vector(32) and infer(SV.binary("+", 1, 2)) == T["integer"]
assert infer(SV.binary("+", SV.cast("byte", 1), SV.cast("int", 1))) == T["int"] and infer(SV.binary("*", 1, 1.5)) == T["real"]
assert infer(SV.binary("**", v("1010"), 2)) == SVD.vector(4) and infer(SV.binary("**", 2, 0.5)) == T["real"]
assert infer(SV.binary("<<", v("1010"), 40)) == SVD.vector(4) and infer(SV.binary("==", v("1010"), 1)) == SVD.vector(1)
assert infer(SV.binary("===", v("1010"), 1)) == T["bit"] and infer(SV.binary("==?", v("1010"), 1)) == SVD.vector(1)
assert infer(SV.binary("&&", x, 1), {"x": D.Anything}) == SVD.vector(1) and infer(SV.binary("+", x, 1), {"x": D.Anything}) is D.Anything
assert infer(SV.binary("&", v("10"), v("1000"))) == SVD.vector(4) and infer(SV.unary("~", SV.cast("byte", 1))) == T["byte"]
assert infer(SV.unary("-", 1.5)) == T["real"] and infer(SV.unary("!", 1.5)) == SVD.vector(1) and infer(SV.unary("^", v("10"))) == SVD.vector(1)
assert infer(SV.binary("%", 7, 2)) == T["integer"] and infer(SV.cast("int", x), {"x": D.Anything}) == T["int"]
assert infer(SV.cast(8, x), {"x": D.Anything}) is D.Anything and infer(SV.cast("signed", 1)) is D.Anything  # sized by the operand
assert infer(SV.inside(x, 1), {"x": D.Anything}) == SVD.vector(1) and infer(SV.conditional(x, 1, 2), {"x": D.Anything}) is D.Anything
assert v("12").typed() is None and infer(v("10x1", True)) == SVD.vector(4, True) and infer(SV.constant(1.5)) == T["real"]
assert infer(SV.binary("+", "a", 1)) is D.Anything  # a string has no type of its own
with raises(TypeError, match="+ cannot take (int | real, integer); it takes (T, T) -> their common type"):
    infer(SV.binary("+", x, 1), {"x": D.OfUnion(T["int"], T["real"])})
assert SVD.BINARY["+"].describe() == "(T, T) -> their common type" and SVD.UNARY["!"].arity() == 1
for expression, message in [(SV.binary("%", 1.5, 1), "% cannot take (real, integer); it takes (integral, integral) -> their common type"),
                            (SV.binary("&", 1.5, 1), "& cannot take (real, integer)"), (SV.binary("<<", 1.5, 1), "<< cannot take (real, integer)"),
                            (SV.binary("===", 1.5, 1), "=== cannot take (real, integer)"), (SV.binary("==?", 1.5, 1), "==? cannot take (real, integer)"),
                            (SV.unary("~", 1.5), "~ cannot take (real); it takes (integral) -> its type"), (SV.unary("&", 1.5), "& cannot take (real)")]:
    with raises(TypeError, match=message):
        infer(expression)

## DIA-14 · SystemVerilog evaluates by IEEE 1800's rules: sizing by context, 4-state logic, casts and system functions

In [ ]:
def ev(expression, **variables):
    return SVV.OfAny(expression, variables)


v, b, u, c, call = SV.vector, SV.binary, SV.unary, SV.cast, SV.call
x = SV.identifier("x")
# Each expression and its value: a Logic by its repr, or a real.
cases = [
 (b("+", v("1010"), v("0110")), "4'b0000"), (b("+", v("1111"), 1), "32'b" + "0"*27 + "10000"),
 (b("/", -7, 2), "32'sb" + "1"*30 + "01"), (b("%", -7, 2), "32'sb" + "1"*32), (b("/", 1, 0), "32'sb" + "x"*32),
 (b(">>>", v("11111110", True), 1), "8'sb11111111"), (b(">>>", v("1000"), 1), "4'b0100"), (b(">>", v("1000", True), 1), "4'sb0100"),
 (b("&", v("10x1"), v("0011")), "4'b00x1"), (b("|", v("10z1"), v("1100")), "4'b11x1"), (b("^", v("10x1"), v("0011")), "4'b10x0"),
 (b("~^", v("1001"), v("0011")), "4'b0101"), (b("+", v("10x1"), 1), "32'b" + "x"*32),
 (b("==", v("10x1"), v("10x1")), "1'bx"), (b("===", v("10x1"), v("10x1")), "1'b1"), (b("!==", v("10z1"), v("10x1")), "1'b1"),
 (b("==?", v("1001"), v("1xx1")), "1'b1"), (b("!=?", v("1001"), v("1xx0")), "1'b1"), (b("==?", v("1x01"), v("1001")), "1'bx"),
 (u("&", v("1111")), "1'b1"), (u("|", v("000x")), "1'bx"), (u("^", v("1011")), "1'b1"), (u("~&", v("1101")), "1'b1"),
 (u("~|", v("0000")), "1'b1"), (u("&", v("1x11")), "1'bx"), (u("|", v("0100")), "1'b1"), (u("~^", v("1x11")), "1'bx"), (u("~^", v("1011")), "1'b0"), (u("^", v("1x11")), "1'bx"), (u("&", v("0x11")), "1'b0"),
 (u("-", v("0001")), "4'b1111"), (u("~", v("10x1")), "4'b01x0"), (u("-", v("0x01")), "4'bxxxx"), (u("+", v("0x01")), "4'b0x01"),
 (b("**", 3, -1), "32'sb" + "0"*32), (b("**", 0, -1), "32'sb" + "x"*32), (b("**", -1, -1), "32'sb" + "1"*32),
 (b("**", -1, -2), "32'sb" + "0"*31 + "1"), (b("**", 2, 10), "32'sb" + "0"*21 + "10000000000"), (b("**", 1, -3), "32'sb" + "0"*31 + "1"),
 (b("**", 2.0, 0.5), 2 ** 0.5), (b("/", 1.0, 0), math.inf), (b("**", 2, 0.5), 2 ** 0.5), (b("+", 1, 0.5), 1.5),
 (SV.conditional(v("x"), v("1100"), v("1010")), "4'b1xx0"), (SV.conditional(v("1"), v("1100"), v("1010")), "4'b1100"),
 (SV.conditional(0, v("1100"), v("1010")), "4'b1010"), (SV.conditional(v("z"), 1.5, 1.5), 1.5), (SV.conditional(v("z"), 1.5, 2.5), 0.0),
 (SV.inside(3, 1, SV.span(2, 5)), "1'b1"), (SV.inside(7, 1, SV.span(2, 5)), "1'b0"), (SV.inside(v("1x00"), v("1100")), "1'bx"),
 (SV.inside(v("1x00"), v("1100"), v("1x00")), "1'b1"), (SV.inside(v("1x00"), SV.span(0, 3)), "1'bx"), (SV.inside(v("1x00"), SV.span(8, 15)), "1'bx"),
 (SV.inside(v("1100"), v("1x00")), "1'b1"),
 (c("byte", 300), "8'sb00101100"), (c("byte", 200), "8'sb11001000"), (c("unsigned", -1), "32'b" + "1"*32),
 (c(4, v("11111111")), "4'b1111"), (c("int", 2.5), "32'sb" + "0"*30 + "11"), (c("int", -2.5), "32'sb" + "1"*30 + "01"),
 (c("real", 3), 3.0), (c("shortreal", 0.1), 0.10000000149011612), (c("int", v("10x1")), "32'sb" + "0"*28 + "1001"),
 (c("bit", v("x")), "1'b0"), (c("signed", v("1111")), "4'sb1111"), (c(8, v("1111", True)), "8'sb11111111"), (c(8, v("z1", True)), "8'sbzzzzzzz1"), (c(4, v("x0", True)), "4'sbxxx0"),
 (call("$clog2", 8), "32'sb" + "0"*30 + "11"), (call("$clog2", 9), "32'sb" + "0"*29 + "100"), (call("$clog2", 0), "32'sb" + "0"*32),
 (call("$clog2", v("1x")), "32'sb" + "x"*32), (call("$bits", v("00000000")), "32'sb" + "0"*28 + "1000"), (call("$bits", 1.5), "32'sb" + "0"*25 + "1000000"),
 (call("$countones", v("1011")), "32'sb" + "0"*30 + "11"), (call("$onehot", v("0100")), "1'b1"), (call("$onehot", v("0000")), "1'b0"),
 (call("$onehot0", v("0000")), "1'b1"), (call("$isunknown", v("10x1")), "1'b1"), (call("$isunknown", v("1001")), "1'b0"),
 (call("$signed", v("1111")), "4'sb1111"), (call("$unsigned", v("1111", True)), "4'b1111"),
 (SV.concatenation(v("1010"), v("11")), "6'b101011"), (SV.replication(2, v("10")), "4'b1010"), (SV.concatenation(v("x"), v("1")), "2'bx1"),
 (SV.select(v("1010"), 1), "1'b1"), (SV.select(v("1010"), 4), "1'bx"), (SV.select(v("1010"), v("x")), "1'bx"), (SV.range_(v("1010"), 3, 2), "2'b10"),
 (SV.range_(v("1010"), 5, 2), "4'bxx10"), (SV.select(c("int", v("1010")), 40), "1'b0"),
 (b("+", v("1111", True), v("00000000", True)), "8'sb11111111"), (b("+", v("1111", True), v("00000000")), "8'b00001111"),
 (b("<", -1, v("1")), "1'b0"), (b("<", -1, 1), "1'b1"), (b("==", b("+", v("1111"), v("0001")), v("10000")), "1'b1"),
 (b("==", b("<<", v("1000"), 1), v("10000")), "1'b1"), (b("<<", v("0001"), v("10")), "4'b0100"), (b("<<", v("0001"), v("x")), "4'bxxxx"),
 (b("+", b("<", 2, 1), v("0001")), "4'b0001"), (b("+", b("<", 1, 2), v("0001")), "4'b0010"),
 (u("!", v("0000")), "1'b1"), (u("!", v("00x0")), "1'bx"), (u("!", 1.5), "1'b0"), (u("!", 0.0), "1'b1"), (b(">>>", v("x000", True), 1), "4'sbxx00"), (b("&&", v("x"), 0), "1'b0"), (b("||", v("x"), 1), "1'b1"),
 (b("&&", v("x"), 1), "1'bx"), (b("||", 0, 0), "1'b0"), (b("&&", 0, v("x")), "1'b0"), (b("||", 1, v("x")), "1'b1"),
 (b("->", 0, v("x")), "1'b1"), (b("->", 1, v("x")), "1'bx"), (b("->", 1, 0), "1'b0"), (b("->", v("x"), 1), "1'b1"),
 (b("<->", 1, 1), "1'b1"), (b("<->", 1, 0), "1'b0"), (b("<->", v("x"), 0), "1'bx"),
 (b(">", 1.5, 1), "1'b1"), (b("==", math.nan, math.nan), "1'b0"), (b("!=", math.nan, 1.0), "1'b1"), (b("<", v("x"), 1), "1'bx"),
 (u("-", 1.5), -1.5), (u("+", 1.5), 1.5), (b("-", c("shortreal", 1.5), c("shortreal", 0.25)), 1.25),
 (b("*", 3, 4), "32'sb" + "0"*28 + "1100"), (b("-", 3, 4), "32'sb" + "1"*32), (b("%", 7, -2), "32'sb" + "0"*31 + "1"),
]
for expression, value in cases:
    result = ev(expression)
    assert (repr(result) == value if type(value) is str else type(result) is float and result == value), (SVText.ToText(expression), result)
assert math.isnan(ev(b("**", -8.0, 1 / 3))) and ev(b("**", 0.0, -1.0)) == math.inf and ev(b("**", 10.0, 400.0)) == math.inf
assert ev(b("**", -10.0, 401.0)) == -math.inf and ev(b("**", -10.0, 400.0)) == math.inf and ev(b("**", c("shortreal", 2), c("shortreal", 0.5))) == 1.4142135381698608
assert ev(b("**", -0.0, -1.0)) == -math.inf and ev(b("**", -0.0, -2.0)) == math.inf and ev(b("**", 0.0, -3.0)) == math.inf
assert ev(b("**", 1.0, math.nan)) == 1.0 and ev(b("**", -1.0, math.inf)) == 1.0 and ev(b("**", math.nan, 0.0)) == 1.0  # as C's pow
assert repr(ev(b("/", c("int", 1), c("int", 0)))) == "32'sb" + "0" * 32 and repr(ev(b("&", c("int", 5), v("x")))) == "32'b" + "0" * 31 + "x"
# Variables are typed by their values, Basic's typed values among them.
assert repr(ev(x, x=True)) == "1'b1" and ev(x, x=5).type == SVD.TYPES["integer"] and ev(x, x=2 ** 40).type == SVD.TYPES["longint"]
assert repr(ev(b("<<", 1, v("1" * 64)))) == "32'sb" + "0" * 32 and repr(ev(b(">>>", -4, v("1" * 64)))) == "32'sb" + "1" * 32
assert repr(ev(call("$bits", c("shortreal", 1.5)))) == "32'sb" + "0" * 26 + "100000" and repr(ev(x, x="é")) == "16'b1100001110101001"
assert repr(ev(x, x="AB")) == "16'b0100000101000010" and repr(ev(x, x="")) == "8'b00000000" and ev(x, x=1.5) == 1.5
assert repr(ev(x, x=SVD.Logic.of("1x"))) == "2'b1x" and repr(ev(b("+", x, 1), x=BD.Value(BD.OfInteger.Data(8, False), 200))) == "32'b" + "0" * 24 + "11001001"
assert ev(x, x=BD.Value(BD.OfInteger.Data(4), -1)).type == SVD.vector(4, True, 2) and repr(ev(x, x=BD.Value(BD.OfBits.Data(12), b"\x0a\xbc"))) == "12'b101010111100"
assert repr(ev(x, x=BD.Value(BD.OfIeee1164.Data(4), "10ZU"))) == "4'b10zx"  # a std_logic_vector is a logic vector
assert [repr(ev(x, x=BD.Value(BD.OfIeee1164.Data(), level))) for level in "01ZLHUXW-"] == ["1'b0", "1'b1", "1'bz", "1'b0", "1'b1"] + ["1'bx"] * 4
assert ev(x, x=BD.Value(BD.OfIeee754.Data("binary32"), 1.5)) == 1.5 and ev(x, x={"a": 1}) == {"a": 1}
assert repr(ev(SV.member(x, "age"), x=ann)) == "32'sb" + "0" * 26 + "011110" and repr(ev(SV.member(x, "a"), x={"a": True})) == "1'b1"
assert repr(ev(SV.select(x, 1), x=[1, 2])) == "32'sb" + "0" * 30 + "10" and repr(ev(SV.select(SV.member(x, "a"), 0), x={"a": [False]})) == "1'b0"
Bus = S.OfObject.Builder().name("Bus").ref().properties(text("name"), lambda p: p.name("ports").of(
    lambda t: t.as_indexed(lambda i: i.of(S.OfNative.Data(int))))).create()
store.register(Bus)
assert repr(ev(SV.select(SV.member(x, "ports"), 1), x=store.Bus().name("b").ports([4, 5]).create())) == "32'sb" + "0" * 29 + "101"
assert SVV.OfAny(SV.call("f", x, 1), SVV.Scope({"x": 2}, functions={"f": lambda a, b: a.integer() + b.integer()})).type == SVD.TYPES["integer"]
assert SVV.OfAny(b("+", 1, 1), SVV.Scope()).integer() == 2 and SVV.OfAny(SV.binary("+", 1, 2)).integer() == 3
for (expression, variables), error, message in [
    ((x, {}), FE.NameError, "identifier 'x' is not declared"), ((SV.call("f", 1), {}), FE.NameError, "function 'f' is not declared"),
    ((v("12"), {}), ValueError, "cannot evaluate an invalid expression: a vector's bits must be 0, 1, x and z, at least one, got '12'"),
    ((x, {"x": 2 ** 70}), OverflowError, "integer 1180591620717411303424 is too large for any type"),
    ((x, {"x": BD.Value(BD.OfIeee754.Data("decimal64"), "1")}), TypeError, "decimal64 has no SystemVerilog type"),
    ((x, {"x": BD.Value(BD.OfInteger.Data(), 1)}), TypeError, "int has no SystemVerilog type"),
    ((u("~", 1.5), {}), TypeError, "~ expects an integral operand, got real"), ((u("&", 1.5), {}), TypeError, "the reduction & expects an integral operand, got real"),
    ((b("%", 1.5, 1), {}), TypeError, "% expects integral operands, got real"), ((b("&", 1.5, 1), {}), TypeError, "& expects integral operands, got real"),
    ((b("<<", 1.5, 1), {}), TypeError, "<< expects an integral operand, got real"), ((b("<<", 1, 1.5), {}), TypeError, "<< expects integral operands, got real"),
    ((b("+", x, 1), {"x": {}}), TypeError, "+ expects a number, got dict"), ((c("int", x), {"x": {}}), TypeError, "a cast expects a number, got dict"),
    ((SV.conditional(1, x, 1), {"x": {}}), TypeError, "?: expects a number, got dict"), ((u("!", x), {"x": {}}), TypeError, "a dict has no truth value"),
    ((c("signed", 1.5), {}), TypeError, "signed' expects an integral operand, got real"), ((c("int", math.nan), {}), OverflowError, "nan has no integer value"),
    ((c("int", -math.inf), {}), OverflowError, "-inf has no integer value"),
    ((SV.member(x, "b"), {"x": {"a": 1}}), KeyError, "no member named 'b'"), ((SV.member(x, "b"), {"x": 1}), TypeError, "a Logic has no members"),
    ((SV.select(x, 2), {"x": [1, 2]}), ValueError, "index 2 is out of bounds of an array of 2"),
    ((SV.select(x, v("x")), {"x": [1, 2]}), ValueError, "an array's index must be known, got 1'bx"),
    ((SV.select(x, 0), {"x": {"a": 1}}), TypeError, "a dict cannot be selected from"), ((SV.range_(x, 1, 0), {"x": {}}), TypeError, "a dict cannot be selected from"),
    ((SV.range_(v("1010"), 0, 1), {}), ValueError, "a range's msb must be at least its lsb, got [0:1]"),
    ((SV.range_(v("1010"), v("x"), 0), {}), ValueError, "a range's msb must be a known integer, got 1'bx"),
    ((SV.range_(v("1010"), 1, 0.5), {}), ValueError, "a range's lsb must be a known integer, got 0.5"),
    ((SV.replication(0, v("1")), {}), ValueError, "a replication's count must be positive, got 0"),
    ((SV.concatenation(v("1"), 1.5), {}), TypeError, "a concatenation's parts must be integral, got float"),
    ((call("$clog2", 1, 2), {}), TypeError, "$clog2 takes 1 argument, got 2"), ((call("$signed", 1.5), {}), TypeError, "$signed expects an integral argument, got real"),
    ((call("$bits", x), {"x": {}}), TypeError, "$bits expects a number, got dict"), ((call("$countones", 1.5), {}), TypeError, "$countones expects an integral argument, got real"),
]:
    with raises(error, match=message):
        ev(expression, **variables)

## DIA-15 · SystemVerilog arrays: methods, reductions with a `with` clause, and `inside` over an array's elements

In [ ]:
xs, p, q = SV.identifier("xs"), SV.identifier("p"), SV.identifier("q")
assert SVText.ToText(SV.iterate(SV.member(xs, "pins"), "and", "p", SV.binary(">", p, 0))) == "xs.pins.and(p) with (p > 0)"
assert SVText.ToText(SV.select(SV.method(xs, "min"), 0)) == "xs.min()[0]" and SVText.ToText(SV.method(SV.binary("+", xs, 1), "size")) == "(xs + 1).size()"
assert SVText.ToText(SV.inside(1, xs, SV.span(2, 3))) == "1 inside {xs, [2:3]}"
assert SV.method(xs, "pop").validate(bound={"xs"}) == [
    "an array method must be one of size, sum, product, and, or, xor, min, max, unique, got 'pop'"]
assert SV.iterate(xs, "min", "p", p).validate(bound={"xs"}) == [  # a malformed iteration binds no name
    "an iteration's method must be one of sum, product, and, or, xor, got 'min'", "body: identifier 'p' is not bound"]
assert SV.iterate(xs, "and", "p", p).validate(bound={"xs"}) == [] and SY.free(SV.iterate(xs, "and", "p", SV.binary("&", p, q))) == {"xs", "q"}
assert SV.DIALECT.infer(SV.iterate(xs, "and", "p", p), {"xs": D.Anything}) is D.Anything
assert SV.DIALECT.infer(SV.method(xs, "size"), {"xs": D.Anything}) is D.Anything


def ev(expression, **variables):
    return SVV.OfAny(expression, variables)


cases = [  # each over [1, 2, 3, 2], and its value
    (SV.method(xs, "size"), "32'sb" + "0" * 29 + "100"), (SV.method(xs, "sum"), "32'sb" + "0" * 28 + "1000"),
    (SV.method(xs, "product"), "32'sb" + "0" * 28 + "1100"), (SV.method(xs, "and"), "32'sb" + "0" * 32),
    (SV.method(xs, "or"), "32'sb" + "0" * 30 + "11"), (SV.method(xs, "xor"), "32'sb" + "0" * 30 + "10"),
    (SV.select(SV.method(xs, "min"), 0), "32'sb" + "0" * 31 + "1"), (SV.select(SV.method(xs, "max"), 0), "32'sb" + "0" * 30 + "11"),
    (SV.method(SV.method(xs, "unique"), "size"), "32'sb" + "0" * 30 + "11"),
    (SV.iterate(xs, "and", "p", SV.binary(">", p, 0)), "1'b1"), (SV.iterate(xs, "and", "p", SV.binary(">", p, 1)), "1'b0"),
    (SV.iterate(xs, "or", "p", SV.binary(">", p, 2)), "1'b1"), (SV.iterate(xs, "xor", "p", SV.binary("==", p, 2)), "1'b0"),
    (SV.iterate(xs, "sum", "p", SV.cast("int", SV.binary(">=", p, 2))), "32'sb" + "0" * 30 + "11"),
    (SV.iterate(xs, "sum", "p", SV.binary("*", p, p)), "32'sb" + "0" * 27 + "10010"),
    (SV.iterate(xs, "product", "p", SV.binary("+", p, 1)), "32'sb" + "0" * 25 + "1001000"),
    (SV.inside(3, xs), "1'b1"), (SV.inside(5, xs, 7), "1'b0"), (SV.inside(7, xs, 7), "1'b1"), (SV.inside(2, SV.span(0, 1), xs), "1'b1"),
]
for expression, value in cases:
    assert repr(ev(expression, xs=[1, 2, 3, 2])) == value, (SVText.ToText(expression), ev(expression, xs=[1, 2, 3, 2]))
# x and z reduce as the operators take them: and() is 0 if any value is, x if any is unknown and none 0.
assert repr(ev(SV.iterate(xs, "and", "p", p), xs=[SVD.Logic.of("1"), SVD.Logic.of("x")])) == "1'bx"
assert repr(ev(SV.iterate(xs, "and", "p", p), xs=[SVD.Logic.of("0"), SVD.Logic.of("x")])) == "1'b0"
assert repr(ev(SV.inside(SV.vector("1x"), xs), xs=[SVD.Logic.of("10")])) == "1'bx"
assert repr(ev(SV.method(SV.method(xs, "unique"), "size"), xs=[SVD.Logic.of("1x"), SVD.Logic.of("1x"), 1.0, 1, 1.0])) == "32'sb" + "0" * 30 + "11"
# Of no items, the reductions give their identities, and the locators no item.
assert [repr(ev(SV.method(xs, m), xs=[])) for m in ("sum", "product", "and", "or", "xor")] == [
    "32'sb" + "0" * 32, "32'sb" + "0" * 31 + "1", "1'b1", "1'b0", "1'b0"]
assert ev(SV.method(xs, "min"), xs=[]) == [] and repr(ev(SV.iterate(xs, "and", "p", SV.binary(">", p, 0)), xs=[])) == "1'b1"
assert ev(SV.method(xs, "sum"), xs=[1.5, 2]) == 3.5 and repr(ev(SV.select(SV.method(xs, "max"), 0), xs=[1.5, 2])) == "32'sb" + "0" * 30 + "10"
for (expression, variables), error, message in [
    ((SV.method(xs, "size"), {"xs": 5}), TypeError, "size() is a method of arrays, not of Logic"),
    ((SV.iterate(xs, "and", "p", p), {"xs": {"a": 1}}), TypeError, "and() is a method of arrays, not of dict"),
    ((SV.method(xs, "sum"), {"xs": [{"a": 1}]}), TypeError, "an array's elements must be numbers, got dict"),
    ((SV.inside(1, xs), {"xs": [{"a": 1}]}), TypeError, "an array's elements must be numbers, got dict"),
    ((SV.inside(1, xs), {"xs": {"a": 1}}), TypeError, "inside expects a number, got dict"),
    ((SV.method(xs, "and"), {"xs": [1.5, 2.5]}), TypeError, "& expects integral operands, got real"),
]:
    with raises(error, match=message):
        ev(expression, **variables)

## DIA-16 · Matlab arrays: arrayfun, indexing from 1, and the functions of arrays

In [ ]:
xs, p = M.identifier("xs"), M.identifier("p")
assert MText.ToText(M.call("all", M.arrayfun("p", M.field(xs, "pins"), M.binary(">", p, 0)))) == "all(arrayfun(@(p) p > 0, xs.pins))"
assert MText.ToText(M.index(xs, M.binary("+", p, 1))) == "xs(p + 1)" and MText.ToText(M.index(M.field(xs, "a"), 2)) == "xs.a(2)"
assert M.arrayfun("_p", xs, 1).validate(bound={"xs"}) == ["an arrayfun's parameter must be an identifier, got '_p'"]
assert M.arrayfun("p", xs, p).validate(bound={"xs"}) == [] and SY.free(M.arrayfun("p", xs, M.binary("+", p, M.identifier("q")))) == {"xs", "q"}
assert M.call("numel", xs, 1).validate(bound={"xs"}) == ["numel takes 1 arguments, got 2"]
assert M.DIALECT.infer(M.call("all", xs), {"xs": D.Anything}).name() == "logical"
assert M.DIALECT.infer(M.arrayfun("p", xs, p), {"xs": D.Anything}) is D.Anything


def alike(a, b):
    if type(b) is list:
        return type(a) is list and len(a) == len(b) and all(map(alike, a, b))
    return type(a) is type(b) and (a == b or (a != a and b != b))


VALUES = [
    (M.call("all", M.arrayfun("p", xs, M.binary(">", p, 0))), [1, 2, 3, 2], True), (M.call("any", M.arrayfun("p", xs, M.binary(">", p, 2))), [1, 2, 3, 2], True),
    (M.call("nnz", M.arrayfun("p", xs, M.binary(">=", p, 2))), [1, 2, 3, 2], 3.0), (M.call("numel", xs), [1, 2, 3, 2], 4.0),
    (M.index(xs, 2), [1, 2, 3, 2], 2.0), (M.index(xs, True), [1, 2, 3, 2], 1.0), (M.index(xs, False), [1, 2, 3, 2], []),
    (M.call("ismember", 3, xs), [1, 2, 3, 2], True), (M.call("ismember", "3", xs), [1, 2, 3, 2], True), (M.call("ismember", 5, xs), [1, 2, 3, 2], False),
    (M.call("sum", xs), [1, 2, 3, 2], 8.0), (M.call("min", xs), [1, 2, 3, 2], 1.0), (M.call("max", xs), [1, 2, 3, 2], 3.0),
    (M.call("numel", M.call("unique", xs)), [1, 2, 3, 2], 3.0), (M.arrayfun("p", xs, M.binary(".*", p, 2)), [1, 2, 3, 2], [2.0, 4.0, 6.0, 4.0]),
    (M.arrayfun("p", xs, M.binary(">", p, 1)), [1, 2, 3, 2], [False, True, True, True]), (M.arrayfun("p", xs, p), [True, 2], [1.0, 2.0]),
    (M.call("all", xs), [], True), (M.call("any", xs), [], False), (M.call("sum", xs), [], 0.0), (M.call("min", xs), [], []),
    (M.call("max", xs), [math.nan, 2], 2.0), (M.call("any", xs), [math.nan], False), (M.call("all", xs), [math.nan], True),
    (M.call("nnz", xs), [math.nan, 0], 1.0), (M.call("min", xs), [True, False], False), (M.call("sum", xs), [True, 2], 3.0),
    (M.call("unique", xs), ["b", "a", "b"], ["a", "b"]), (M.call("unique", xs), [True, True], [True]), (M.call("unique", xs), [3, 1, 3], [1.0, 3.0]),
    (M.call("numel", xs), 5, 1.0), (M.index(xs, 1), 5, 5.0), (M.arrayfun("p", xs, M.binary("+", p, 1)), 5, [6.0]),
    (M.call("ismember", 1, xs), [True], True), (M.call("numel", xs), "abc", 1.0), (M.call("numel", xs), {"a": 1}, 1.0),
]
ERRORS = [
    (M.index(xs, 0), [1, 2], ValueError, "Array indices must be positive integers or logical values."),
    (M.index(xs, 1.5), [1, 2], ValueError, "Array indices must be positive integers or logical values."),
    (M.index(xs, "a"), [1, 2], ValueError, "Array indices must be positive integers or logical values."),
    (M.index(xs, 5), [1, 2, 3, 2], ValueError, "Index exceeds the number of array elements. Index must not exceed 4."),
    (M.arrayfun("p", xs, "a"), [1, 2], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."),
    (M.arrayfun("p", xs, M.binary("+", xs, 0)), [1, 2], TypeError, "Operator '+' is not supported for operands of type 'array' and 'double'."),
    (M.arrayfun("p", xs, xs), [1, 2], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."),
    (M.call("sum", xs), ["a"], TypeError, "Invalid data type. First argument must be numeric or logical."),
    (M.call("all", xs), {"a": 1}, TypeError, "Invalid data type. First argument must be numeric or logical."),
    (M.call("unique", xs), [{"a": 1}], TypeError, "Invalid data type. First argument must be numeric or logical."),
    (M.call("ismember", xs, xs), [1], TypeError, "ismember takes a scalar and an array of scalars."),
    (M.binary("==", xs, 1), [1], TypeError, "Operator '==' is not supported for operands of type 'array' and 'double'."),
    (M.binary("<", xs, 1), {"a": 1}, TypeError, "Operator '<' is not supported for operands of type 'struct' and 'double'."),
]
for expression, value, expected in VALUES:
    result = MV.OfAny(expression, {"xs": value})
    assert alike(result, expected), (MText.ToText(expression), value, result)
for expression, value, error, message in ERRORS:
    with raises(error, match=message):
        MV.OfAny(expression, {"xs": value})
assert math.isnan(MV.OfAny(M.call("max", xs), {"xs": [math.nan]})) and alike(MV.OfAny(M.call("unique", xs), {"xs": [math.nan, 1, math.nan]}), [1.0, math.nan, math.nan])
Crew = S.OfObject.Builder().name("Crew").ref().properties(lambda q: q.name("ranks").of(
    lambda t: t.as_indexed(lambda i: i.of(S.OfNative.Data(int))))).create()
store.register(Crew)
crew = store.Crew().ranks([1, 2, 3]).create()  # a list property is an array of doubles
assert alike(MV.OfAny(M.field(xs, "ranks"), {"xs": crew}), [1.0, 2.0, 3.0]) and MV.OfAny(M.call("sum", M.field(xs, "ranks")), {"xs": crew}) == 6.0

## DIA-17 · Excel arrays: MAP and LAMBDA, AND and OR of any number of arguments, and the functions of arrays

In [ ]:
xs, p = X.name("xs"), X.name("p")
assert XText.ToText(X.function("AND", X.map_("p", X.field(xs, "pins"), X.infix(">", p, 0)))) == "=AND(MAP(xs.pins, LAMBDA(p, p > 0)))"
assert XText.ToText(X.function("INDEX", xs, X.infix("+", p, 1))) == "=INDEX(xs, p + 1)"
assert X.map_("p", xs, p).validate(bound={"xs"}) == [] and SY.free(X.map_("p", xs, X.infix("+", p, X.name("q")))) == {"xs", "q"}
assert X.function("AND", 1, 2, 3).validate() == [] and X.function("ROWS", xs, 1).validate(bound={"xs"}) == ["ROWS takes 1 arguments, got 2"]
assert X.DIALECT.infer(X.function("AND", xs), {"xs": D.Anything}).name() == "logical"
assert X.DIALECT.infer(X.map_("p", xs, p), {"xs": D.Anything}) is D.Anything
assert XV.OfAny(X.function("AND", True, True, False)) is False and XV.OfAny(X.function("OR", False, False, True)) is True
# IF takes 2 or 3 arguments, and AND, OR, SUM, MIN and MAX 1 to 255, as Excel's do.
assert X.function("IF", True, 1).validate() == [] and XV.OfAny(X.function("IF", False, 1)) is False and XV.OfAny(X.function("IF", True, 1)) == 1.0
assert X.function("IF", True).validate() == ["IF takes 2 or 3 arguments, got 1"] and X.function("SUM").validate() == ["SUM takes 1 to 255 arguments, got 0"]
assert XV.OfAny(X.function("SUM", 1, xs, True), {"xs": [2, "a", True]}) == 4.0 and XV.OfAny(X.function("MAX", xs, 7), {"xs": [1, 2]}) == 7.0
assert XV.OfAny(X.function("MIN", 3, X.name("e")), {"e": XV.Error("#N/A")}) == XV.Error("#N/A")
for expression, message in [(X.function("IF", True, 1, 2, 3), "IF takes 2 or 3 arguments, got 4"), (X.function("AND"), "AND takes 1 to 255 arguments, got 0"),
                            (X.function("MIN"), "MIN takes 1 to 255 arguments, got 0")]:
    with raises(TypeError, match=message):
        XV.OfAny(expression)
VALUES = [
    (X.function("AND", X.map_("p", xs, X.infix(">", p, 0))), [1, 2, 3, 2], True), (X.function("OR", X.map_("p", xs, X.infix(">", p, 2))), [1, 2, 3, 2], True),
    (X.function("SUM", X.map_("p", xs, X.function("IF", X.infix(">=", p, 2), 1, 0))), [1, 2, 3, 2], 3.0), (X.function("ROWS", xs), [1, 2, 3, 2], 4.0),
    (X.function("INDEX", xs, 2), [1, 2, 3, 2], 2.0), (X.function("INDEX", xs, 2.9), [1, 2, 3, 2], 2.0), (X.function("INDEX", xs, 5), [1, 2], XV.Error("#REF!")),
    (X.function("INDEX", xs, 0), [1, 2], XV.Error("#REF!")), (X.function("INDEX", xs, "x"), [1, 2], XV.Error("#VALUE!")),
    (X.function("MATCH", 3, xs, 0), [1, 2, 3, 2], 3.0), (X.function("MATCH", "B", xs, 0), ["a", "b"], 2.0), (X.function("MATCH", 5, xs, 0), [1, 2], XV.Error("#N/A")),
    (X.function("MATCH", "1", xs, 0), [1, 2], XV.Error("#N/A")), (X.function("MATCH", 1, xs, 1), [1, 2], XV.Error("#VALUE!")),
    (X.function("MATCH", True, xs, 0), [1, True], 2.0), (X.function("ISNUMBER", X.function("MATCH", 2, xs, 0)), [1, 2], True),
    (X.function("ISNUMBER", X.function("MATCH", 7, xs, 0)), [1, 2], False), (X.function("ISNUMBER", "1"), [], False),
    (X.function("SUM", xs), [1, 2, 3, 2], 8.0), (X.function("SUM", xs), [1, "a", True, 2], 3.0), (X.function("SUM", xs), [1, XV.Error("#N/A")], XV.Error("#N/A")),
    (X.function("SUM", True), [], 1.0), (X.function("SUM", "2"), [], 2.0), (X.function("SUM", "a"), [], XV.Error("#VALUE!")),
    (X.function("MIN", xs), [3, 1, 2], 1.0), (X.function("MAX", xs), [3, 1, 2], 3.0), (X.function("MAX", xs), ["a"], 0.0), (X.function("MIN", xs), [], 0.0),
    (X.function("ROWS", X.function("UNIQUE", xs)), [1, 2, 3, 2], 3.0), (X.function("ROWS", X.function("UNIQUE", xs)), ["a", "A", 1, True, "1"], 4.0),
    (X.function("ROWS", X.function("UNIQUE", xs)), [XV.Error("#N/A"), XV.Error("#N/A"), {"a": 1}, {"a": 1}], 3.0),
    (X.function("ROWS", xs), 5, 1.0), (X.function("INDEX", xs, 1), 5, 5.0), (X.function("ROWS", X.map_("p", xs, X.infix("*", p, 2))), 5, 1.0),
    (X.function("AND", xs), [True, "a", 1], True), (X.function("AND", xs), ["a"], XV.Error("#VALUE!")), (X.function("OR", xs), [], XV.Error("#VALUE!")),
    (X.function("AND", xs, False), [True], False), (X.function("OR", xs), [0, XV.Error("#DIV/0!")], XV.Error("#DIV/0!")),
    (X.function("INDEX", X.map_("p", xs, X.infix("-", p, "a")), 1), [1], XV.Error("#VALUE!")), (X.function("ROWS", X.map_("p", xs, p)), XV.Error("#REF!"), XV.Error("#REF!")),
    (X.infix("+", xs, 1), [1], XV.Error("#VALUE!")), (X.infix("=", xs, 1), [1], XV.Error("#VALUE!")),
    (X.function("MATCH", 1, xs, "a"), [1], XV.Error("#VALUE!")), (X.function("UNIQUE", xs), XV.Error("#REF!"), XV.Error("#REF!")),
]
assert XV.OfAny(X.function("INDEX", xs, X.name("n")), {"xs": [1], "n": math.inf}) == XV.Error("#NUM!")
for expression, value, expected in VALUES:
    result = XV.OfAny(expression, {"xs": value})
    assert result == expected and type(result) is type(expected), (XText.ToText(expression), value, result)
Team = S.OfObject.Builder().name("Team").ref().properties(lambda q: q.name("ranks").of(
    lambda t: t.as_indexed(lambda i: i.of(S.OfNative.Data(int))))).create()
store.register(Team)
team = store.Team().ranks([1, 2, 3]).create()  # a list property is an array of numbers
assert XV.OfAny(X.field(xs, "ranks"), {"xs": team}) == [1.0, 2.0, 3.0] and XV.OfAny(X.function("SUM", X.field(xs, "ranks")), {"xs": team}) == 6.0
assert XV.OfAny(X.map_("p", xs, X.infix(">", p, 1)), {"xs": [1, 2]}) == [False, True]